# Practica 2.1
# Equipo los Morgue
# 7BM1

- García Martínez Carla Edith 
- Mena Fernández Sofia 
- Suárez Ramírez Gustavo 
- Varillas Figueroa Enrique Uriel 
- García Carrizoza André

# 1. Crear las carpetas del proyecto


In [ ]:
# Importamos la clase Path del módulo nativo pathlib. 
# Esto nos permite manejar y manipular rutas de archivos y directorios 
# de forma orientada a objetos (independientemente de si es Windows, Mac o Linux).
from pathlib import Path

# Definimos la ruta absoluta de la carpeta principal del proyecto.
# Path.cwd() obtiene el directorio de trabajo actual (Current Working Directory).
# El operador '/' está sobrecargado en pathlib para concatenar carpetas fácilmente.
carpeta_proyecto = Path.cwd() / "Reconocimiento_Locutor"  

# Definimos las rutas para las subcarpetas específicas del proyecto.
# A partir de la carpeta principal, creamos una para los datos y otra para los resultados.
carpeta_datos = carpeta_proyecto / "datos"  
carpeta_resultados = carpeta_proyecto / "resultados"  

# Ejecutamos la creación física de los directorios en el disco duro.
# El método .mkdir() crea la carpeta.
# parents=True: Asegura que si una carpeta padre en la ruta no existe, también la cree.
# exist_ok=True: Evita que el código lance un error (FileExistsError) si la carpeta ya había sido creada antes.
carpeta_proyecto.mkdir(parents=True, exist_ok=True)  
carpeta_datos.mkdir(parents=True, exist_ok=True)  
carpeta_resultados.mkdir(parents=True, exist_ok=True)  

# Imprimimos en la salida estándar (consola) las rutas finales.
# Esto sirve como validación visual para asegurar que las variables apuntan a donde esperamos.
print("Carpeta del proyecto:", carpeta_proyecto)
print("Carpeta de datos:", carpeta_datos)
print("Carpeta de resultados:", carpeta_resultados)

# 2. Descargar el subconjunto de LibriSpeech dev-clean

In [ ]:
# Importamos el módulo urllib.request, que pertenece a la biblioteca estándar de Python.
# Este módulo contiene funciones y clases que nos permiten abrir y leer URLs,
# lo cual es ideal para realizar descargas de archivos desde internet.
import urllib.request

# Definimos la URL directa desde donde vamos a descargar nuestro conjunto de datos de audio.
url_descarga = ("https://openslr.elda.org/resources/12/dev-clean.tar.gz")

# Definimos las rutas de destino utilizando la variable 'carpeta_datos' (creada en la celda anterior).
# archivo_comprimido: Será el nombre y ruta final de nuestro archivo una vez descargado completamente.
archivo_comprimido = carpeta_datos / "dev-clean.tar.gz"  
# archivo_temporal: Archivo intermedio que se usará mientras los datos se están bajando.
# Esto es una buena práctica para evitar tener un archivo corrupto si la descarga se interrumpe.
archivo_temporal = carpeta_datos / "dev-clean.tar.gz.part" 

# Comprobamos si el archivo final ya existe en nuestro disco duro mediante el método .exists() de pathlib.
if archivo_comprimido.exists():
    # Si existe, evitamos volver a descargarlo (ahorrando tiempo y ancho de banda).
    print("El archivo ya existe:", archivo_comprimido) 
else:
    # Si no existe, comenzamos el proceso de descarga.
    print("Descargando LibriSpeech dev-clean...")

    # Utilizamos un bloque try-except para manejar posibles errores de conexión o escritura.
    try:
        # urllib.request.urlretrieve obtiene el contenido de la URL y lo guarda directamente en disco.
        # Aquí lo guardamos primero bajo el nombre temporal (.part).
        urllib.request.urlretrieve(url_descarga, archivo_temporal) 

        # Una vez que la descarga de urlretrieve finaliza con éxito, 
        # reemplazamos/renombramos el archivo temporal para que adquiera su nombre definitivo.
        archivo_temporal.replace(archivo_comprimido) 
        print("Descarga finalizada.")

    # Si ocurre cualquier excepción (error de red, tiempo de espera, disco lleno, etc.).
    except Exception:
        print("La descarga no terminó")
        # El comando 'raise' relanza el error original para que la ejecución se detenga 
        # y podamos ver en la consola exactamente qué falló.
        raise  

# Una vez asegurada la existencia del archivo (ya sea bajado ahora o previamente),
# usamos .stat() que devuelve información del sistema sobre el archivo.
# .st_size extrae el peso exacto del archivo en bytes.
tamano_bytes = archivo_comprimido.stat().st_size  

# Convertimos los bytes a Megabytes (MB) para que sea humanamente legible,
# dividiendo la cantidad de bytes entre 1,000,000.
tamano_nb = tamano_bytes / 1_000_000  

# Imprimimos la ruta del archivo y su tamaño calculado.
print("Archivo:", archivo_comprimido)
# Utilizamos f-strings para formatear el texto. 
# {tamano_nb:.2f} indica que queremos mostrar la variable flotante con solo 2 decimales.
print(f"Tamaño: {tamano_nb:.2f} MB")

# 3. Verificar la integridad del archivo descargado

In [ ]:
# Se importa la biblioteca hashlib para aplicar algoritmos de hash seguros (como MD5).
# Esto nos servirá para crear una "huella digital" de nuestro archivo descargado.
import hashlib 
# Se importa la biblioteca urllib.request para abrir URLs y leer datos directamente desde internet.
import urllib.request 

# Se define la URL que contiene un archivo de texto con las firmas (hashes) oficiales del dataset.
url_verificacion = "https://openslr.elda.org/resources/12/md5sum.txt"  

# Usamos el bloque 'with' para abrir la URL de manera segura, asegurando que la conexión se cierre al terminar.
# Se añade un timeout de 30 segundos para evitar que el programa se quede congelado si no hay respuesta del servidor.
with urllib.request.urlopen(url_verificacion, timeout=30) as respuesta: 
    # Leemos la respuesta HTTP cruda en bytes.
    contenido = respuesta.read()

# Se decodifica el contenido en bytes a una cadena de texto (string) utilizando codificación UTF-8, 
# que es el estándar para texto en internet.
texto_verificacion = contenido.decode("utf-8")  

# Inicializamos la variable 'md5_esperado' como 'None'. 
# Aquí guardaremos el hash oficial una vez que lo encontremos en el texto.
md5_esperado = None 

# Iteramos sobre el texto descargado línea por línea.
# El método .splitlines() separa el texto gigantesco en una lista de líneas individuales.
for linea in texto_verificacion.splitlines(): 
    # .split() divide la línea en una lista usando los espacios en blanco como separador.
    # Usualmente las líneas tienen el formato: "hash_md5 nombre_del_archivo"
    campos = linea.split()  

    # Verificamos que la línea realmente contenga dos elementos (el hash y el nombre).
    # Esto evita errores si el archivo de texto tiene líneas vacías o mal formateadas.
    if len(campos) != 2:  
        continue  # Si no tiene 2 campos, saltamos esta iteración y pasamos a la siguiente línea.

    # Extraemos el nombre del archivo (segundo campo en la lista).
    # .lstrip("*") elimina el asterisco inicial si es que el archivo de verificación lo incluye.
    nombre_archivo = campos[1].lstrip("*")  
    # Si la ruta tiene directorios (ej. carpeta/archivo.tar.gz), nos quedamos solo con el nombre del archivo.
    nombre_archivo = nombre_archivo.split("/")[-1]  

    # Comparamos el nombre que acabamos de leer con el nombre de nuestro archivo local.
    # archivo_comprimido.name extrae solo el nombre del archivo de la ruta en pathlib (dev-clean.tar.gz).
    if nombre_archivo == archivo_comprimido.name:  
        # Si coinciden, extraemos el primer campo (el hash oficial) y lo convertimos a minúsculas por seguridad.
        md5_esperado = campos[0].lower()  
        # Como ya encontramos lo que buscábamos, rompemos el ciclo 'for' para no gastar más tiempo de procesamiento.
        break  
    
# Validamos si logramos encontrar el hash. Si sigue siendo 'None', significa que nuestro archivo 
# no estaba listado en el documento oficial, por lo que lanzamos un error que detiene el código.
if md5_esperado is None:  
    raise ValueError("No se encontró la suma de comprobación de dev-clean.")

# Creamos un objeto 'verificador' con el algoritmo MD5 para calcular el hash de nuestro archivo local.
verificador = hashlib.md5()  
# Definimos un tamaño de bloque de 1 Megabyte (1024 bytes * 1024 bytes).
# Es crucial leer el archivo en trozos (bloques) para evitar cargar todo de golpe en la memoria RAM.
tamano_bloque = 1024*1024  

# Abrimos el archivo comprimido que descargamos en modo "rb" (read binary / lectura binaria).
# Los archivos como imágenes, videos o comprimidos deben leerse en binario, no como texto.
with archivo_comprimido.open("rb") as archivo: 
    # Iniciamos un bucle infinito para leer el archivo por fragmentos.
    while True:  
        # Leemos exactamente 1 Megabyte del archivo.
        bloque = archivo.read(tamano_bloque) 

        # Si 'bloque' está vacío (False), significa que llegamos al final del archivo.
        if not bloque:  
            break  # Salimos del bucle infinito.

        # Alimentamos el objeto verificador con el fragmento leído.
        # Esto va actualizando el cálculo del hash en tiempo real sin desbordar la memoria.
        verificador.update(bloque)  

# Una vez que procesamos todo el archivo, extraemos el resultado final del hash.
# .hexdigest() convierte ese resultado binario en una cadena de texto hexadecimal legible.
md5_calculado = verificador.hexdigest() 

# Mostramos ambos hashes en pantalla para comparación visual.
print("MD5 esperado: ", md5_esperado)
print("MD5 calculado: ", md5_calculado)

# Creamos una variable booleana que será 'True' si ambos hashes son exactamente iguales, o 'False' si difieren.
integridad_correcta = md5_calculado == md5_esperado

# Si la variable es 'False', la integridad falló (el archivo está corrupto o incompleto).
# Lanzamos una excepción para abortar la ejecución, evitando extraer basura.
if not integridad_correcta:
    raise ValueError("La verificación falló. No extraigas el archivo todavía")

# Si el código llega a esta línea sin lanzar errores, la descarga es perfecta.
print("Verificación correcta. El archivo está listo para extraerse.")

# 4. Extraer el conjunto de datos LibriSpeech dev-clean

In [ ]:
# Se importa el módulo nativo tarfile, diseñado para leer y escribir archivos 
# empaquetados (.tar) y comprimidos (como .tar.gz o .tgz).
import tarfile 

# Se realiza una validación de seguridad inicial. Depende de la variable booleana 
# 'integridad_correcta' creada en la celda anterior. Si el archivo descargado 
# estaba corrupto, el programa se aborta aquí antes de intentar extraer basura.
if not integridad_correcta:
    raise ValueError("El archivo debe pasar la verificación antes de extraerse.")

# Definimos la ruta específica donde esperamos encontrar los datos de audio 
# una vez que la extracción finalice correctamente.
carpeta_corpus = carpeta_datos / "LibriSpeech" / "dev-clean"

# Usamos .resolve() para obtener la ruta absoluta y real del directorio de datos.
# Esto es muy importante como medida de seguridad (para evitar el ataque "Zip Slip")
# asegurando que los archivos no se extraigan fuera de esta carpeta base.
destino_base = carpeta_datos.resolve()

# Inicializamos un contador para saber cuántos archivos ya fueron extraídos previamente.
archivos_existentes = 0
# Creamos una lista vacía donde almacenaremos únicamente los archivos que faltan por extraer.
miembros_pendientes = []

# Abrimos el archivo comprimido usando el bloque 'with' (para que se cierre automáticamente).
# mode="r:gz" indica que vamos a Leer (r) un archivo comprimido con Gzip (gz).
with tarfile.open(archivo_comprimido, mode= "r:gz") as archivo:
    # Obtenemos la lista completa de todo lo que contiene el archivo comprimido 
    # (carpetas, subcarpetas, archivos de texto, audios, etc.).
    miembros = archivo.getmembers()

    # Iniciamos un bucle para revisar, uno por uno, cada elemento dentro del archivo.
    for miembro in miembros:
        # Construimos la ruta donde este elemento específico debería guardarse en nuestro disco duro
        # y calculamos su ruta absoluta con .resolve().
        ruta_destino = (carpeta_datos / miembro.name).resolve()

        # VALIDACIÓN DE SEGURIDAD (Prevención de Path Traversal):
        # Verificamos que la ruta de destino planeada esté estrictamente contenida dentro 
        # de nuestra carpeta 'destino_base'. Esto bloquea archivos maliciosos que tengan 
        # nombres como "../../../etc/password".
        if destino_base not in ruta_destino.parents:
            raise ValueError(f"Ruta de extracción no permitida:  {miembro.name}")

        # Si el elemento que estamos revisando es un directorio (carpeta):
        if miembro.isdir():
            # Creamos la carpeta físicamente en el disco duro.
            ruta_destino.mkdir(parents = True, exist_ok = True)
            continue # Saltamos a la siguiente iteración del bucle, ya que las carpetas no se extraen como archivos.

        # Si el elemento no es una carpeta, validamos que sí sea un archivo normal (isfile).
        # Esto filtra enlaces simbólicos u otros elementos raros que no nos interesan.
        if not miembro.isfile():
            raise ValueError(f"Tipo de archivo no permitido: {miembro.name}")

        # Comprobamos si el archivo ya existe físicamente en nuestro disco duro (de una extracción pasada).
        if ruta_destino.exists():
            # Si existe, consultamos su tamaño actual en bytes en nuestro disco duro.
            tamano_actual = ruta_destino.stat().st_size

            # Lo comparamos con el tamaño que el archivo declara tener dentro del .tar.gz (miembro.size).
            # Si los tamaños no coinciden, el archivo existente está incompleto o corrupto.
            if tamano_actual != miembro.size:
                raise ValueError(f"El tamaño del archivo no coincide: {ruta_destino}")

            # Si el tamaño coincide, asumimos que el archivo ya está extraído correctamente y aumentamos el contador.
            archivos_existentes += 1
        
        else:
            # Si el archivo NO existe en nuestro disco duro, lo añadimos a la lista de pendientes por extraer.
            miembros_pendientes.append(miembro)

    # Una vez revisados todos los elementos, comprobamos si la lista de pendientes tiene algo.
    if miembros_pendientes:
        print("Extratendo los archivos faltantes...")

        # Ordenamos la extracción masiva, pero pasándole ÚNICAMENTE la lista de los archivos 
        # que nos faltan (miembros_pendientes). Esto ahorra muchísimo tiempo si el proceso 
        # se había interrumpido antes y lo volvemos a ejecutar.
        archivo.extractall(
            path= carpeta_datos,
            members = miembros_pendientes)

# Finalizada la extracción, verificamos que la estructura principal de carpetas del dataset 
# (LibriSpeech/dev-clean) realmente exista y sea un directorio válido.
if not carpeta_corpus.is_dir():
    raise FileNotFoundError("No se encontró la carpeta del corpus.")

# Imprimimos un resumen en consola de lo que el algoritmo hizo.
print("Archivos existentes con tamaño correcto:", archivos_existentes)
print("Archivos extraídos: ", len(miembros_pendientes))
print("Comprobación de extracción finalizada.")

# 5. Contar los locutores y las grabaciones

In [ ]:
# Verificamos nuevamente que la carpeta del corpus exista y sea un directorio válido.
# Si no existe, lanzamos un error y detenemos la ejecución.
if not carpeta_corpus.is_dir():
    raise FileNotFoundError("No se encontó la carpeta del corpus")

# Creamos una lista vacía para almacenar las rutas de las carpetas correspondientes a cada locutor.
carpetas_locutores = []

# Iteramos sobre todos los elementos dentro de la carpeta del corpus usando .iterdir()
for carpeta in carpeta_corpus.iterdir():
    # Filtramos: solo nos interesan los elementos que sean directorios (carpetas) 
    # Y cuyo nombre esté compuesto únicamente por números (.isdigit()). 
    # En LibriSpeech, cada locutor tiene una carpeta nombrada con su ID numérico.
    if carpeta.is_dir() and carpeta.name.isdigit():
        carpetas_locutores.append(carpeta)

# Definimos una función auxiliar que toma la ruta de una carpeta y devuelve su nombre 
# convertido a un número entero. Esto nos servirá para ordenar las carpetas numéricamente.
def obtener_identificador(carpeta):
    return int(carpeta.name)

# Ordenamos la lista de carpetas de locutores utilizando la función anterior como criterio (key).
# Así, la carpeta "2" irá antes que la "10", en lugar de ordenarse alfabéticamente.
carpetas_locutores.sort(key = obtener_identificador)

# Inicializamos una lista para guardar un diccionario con la información de cada locutor.
resumen_locutores = []
# Inicializamos un contador global para saber cuántas grabaciones hay en total en todo el dataset.
total_grabaciones = 0

# Iniciamos un bucle para procesar cada carpeta de locutor (ya ordenadas).
for carpeta_locutor in carpetas_locutores:
    # Usamos .rglob("*.flac") (recursive glob) para buscar todos los archivos que terminen en .flac 
    # dentro de la carpeta del locutor y en todas sus posibles subcarpetas.
    # Convertimos el resultado a una lista para poder contar sus elementos.
    archivos_locutor = list(carpeta_locutor.rglob("*.flac"))
    
    # Obtenemos la cantidad de grabaciones (archivos .flac) que tiene este locutor específico.
    cantidad_grabaciones = len(archivos_locutor)

    # Creamos un diccionario llamado 'registro' para organizar la información de este locutor.
    registro ={
        "locutor": carpeta_locutor.name, # El ID del locutor (nombre de la carpeta)
        "grabaciones": cantidad_grabaciones, # Cuántos audios tiene
    }

    # Agregamos este diccionario a nuestra lista de resumen general.
    resumen_locutores.append(registro)
    # Sumamos las grabaciones de este locutor al contador global.
    total_grabaciones += cantidad_grabaciones

# Obtenemos la cantidad total de locutores basándonos en cuántas carpetas encontramos.
numero_locutores = len(carpetas_locutores)

# Comenzamos a imprimir un reporte en la consola.
print("Número de locutores:", numero_locutores)
# Utilizamos f-strings con formato de alineación para crear el encabezado de una tabla.
# '<12' alinea a la izquierda ocupando 12 espacios, y '>12' alinea a la derecha ocupando 12 espacios.
print(f"{'Locutor':<12}{'Grabaciones':>12}")
# Imprimimos una línea de puntos para separar el encabezado de los datos.
print("."*24)

# Recorremos la lista de diccionarios que creamos anteriormente para imprimir cada fila.
for registro in resumen_locutores:
    identificador = registro["locutor"]
    cantidad = registro["grabaciones"]

    # Imprimimos los datos de cada locutor aplicando el mismo formato de columnas que en el encabezado.
    print(f"{identificador:<12}{cantidad:>12}")

# Validaciones finales de consistencia de los datos:
# El dataset dev-clean de LibriSpeech debe tener exactamente 40 locutores.
# Si encontramos un número diferente, levantamos un error indicando que los datos podrían estar incompletos.
if numero_locutores != 40:
    raise ValueError(f"Se esperaban 40 locutores, pero no se encontraron {numero_locutores}.")

# Si el total de grabaciones es cero, significa que no se extrajeron los audios correctamente.
if total_grabaciones == 0:
    raise ValueError("No se encotraron grabaciones FLAC.")

# 6. Crear el inventario de metadatos de los audios 

In [ ]:
# Importamos pandas, la biblioteca estándar de Python para análisis y manipulación de datos.
# La usamos para crear y manejar estructuras tabulares llamadas DataFrames.
import pandas as pd
# Importamos soundfile, una biblioteca especializada en leer y escribir archivos de audio.
# Nos permitirá extraer la información técnica de los audios sin tener que cargarlos completamente en memoria.
import soundfile as sf

# Importamos la función 'display' específica para entornos como Jupyter Notebooks o Google Colab.
# Permite renderizar y mostrar DataFrames (tablas) de forma mucho más limpia y visual que un simple 'print'.
from IPython.display import display

# Buscamos todos los archivos .flac dentro de la carpeta del corpus de forma recursiva (.rglob).
# Usamos sorted() para ordenar alfabéticamente la lista de rutas obtenidas y mantener la consistencia.
archivos_audio = sorted(carpeta_corpus.rglob("*.flac"))

# Inicializamos dos listas vacías. 
# Una para guardar los metadatos extraídos con éxito y otra para registrar los archivos que fallen.
registros_audio = []
errores_lectura = []

# Iniciamos un bucle para recorrer cada ruta de archivo de audio encontrada.
for ruta_audio in archivos_audio:
    # Usamos un bloque try-except porque leer archivos físicos siempre es propenso a errores 
    # (archivos corruptos, permisos denegados, etc.).
    try:
        # sf.info() lee exclusivamente las cabeceras (metadatos) del archivo de audio, lo cual es muy rápido.
        # Convertimos 'ruta_audio' (objeto Path) a string puro porque soundfile lo requiere así.
        informacion = sf.info(str(ruta_audio))

        # Creamos un diccionario recolectando toda la información que nos interesa del archivo y su estructura.
        registro = {
            "archivo": ruta_audio.name, # El nombre del archivo con su extensión (ej. 1234-5678-0001.flac)
            # La jerarquía de LibriSpeech es: locutor/capítulo/archivo.flac
            # Por lo tanto, el "abuelo" de la ruta del archivo es el ID del locutor.
            "locutor": ruta_audio.parent.parent.name, 
            # El "padre" de la ruta del archivo es el ID del capítulo del audiolibro.
            "capítulo": ruta_audio.parent.name,
            "ruta": str(ruta_audio), # La ruta absoluta en el disco para poder cargar el audio más adelante
            "duracion_segundos": informacion.duration, # Duración en segundos
            "frecuencia_muestreo": informacion.samplerate, # Tasa de muestreo (ej. 16000 Hz)
            "canales": informacion.channels, # Número de canales (1 para Mono, 2 para Estéreo)
            "numero_muestras":informacion.frames, # Cantidad total de puntos de datos en el audio
            "formato": informacion.format, # Formato contenedor (ej. FLAC)
            "subtipo": informacion.subtype # Subtipo de codificación de los datos
        }

        # Añadimos nuestro diccionario con los datos a la lista de registros exitosos.
        registros_audio.append(registro)
        
    # Si ocurre un error del sistema operativo (OSError) o de ejecución (RuntimeError) al leer el archivo:
    except (OSError, RuntimeError) as error:
        # Creamos un diccionario almacenando la ruta que falló y el mensaje del error exacto.
        registro_error = {
            "archivo": str(ruta_audio),
            "error": str(error)
        }

        # Añadimos el registro fallido a nuestra lista de errores.
        errores_lectura.append(registro_error)

# Convertimos la lista de diccionarios exitosos en un DataFrame (una tabla estructurada) usando pandas.
inventario_audio = pd.DataFrame(registros_audio)
# Hacemos lo mismo con la lista de errores, especificando explícitamente el nombre de las columnas.
tabla_errores = pd.DataFrame(
    errores_lectura,
    columns=["archivo","error"])

# Imprimimos un resumen cuantitativo en la consola.
print("Archivos encontrados:",len(archivos_audio))
print("Metadatos leídos:", len(inventario_audio))
print("Errores de lectura:", len(tabla_errores))

# Mostramos en pantalla las primeras 5 filas (.head()) de nuestra tabla principal de audios
# usando la función 'display' para que se vea con un formato agradable de tabla en el notebook.
display(inventario_audio.head())

# Si nuestra tabla de errores NO está vacía, la mostramos en pantalla para analizar qué falló.
if not tabla_errores.empty:
    display(tabla_errores)

# Validación final crítica: si la tabla principal está vacía, significa que falló la lectura 
# de absolutamente todos los archivos. Lanzamos un error deteniendo la ejecución.
if inventario_audio.empty:
    raise ValueError("No se pudieron leer los metadatos de ninguna grabación.")

# 7. Resumir la duración y los formatos del corpus

In [ ]:
# Obtenemos la cantidad total de grabaciones midiendo la longitud (número de filas) 
# de nuestro DataFrame 'inventario_audio'.
cantidad_grabaciones = len(inventario_audio)

# Calculamos el número de locutores únicos.
# Seleccionamos la columna "locutor" del DataFrame y usamos .nunique() (Number of Unique),
# que cuenta cuántos valores diferentes existen en esa columna, sin contar duplicados.
cantidad_locutores = inventario_audio["locutor"].nunique()

# Calculamos la duración total del dataset en segundos.
# Seleccionamos la columna "duracion_segundos" y aplicamos .sum() para sumar todos sus valores.
duracion_total_segundos = inventario_audio["duracion_segundos"].sum()

# Convertimos la duración total de segundos a minutos dividiendo entre 60.
duracion_total_minutos = duracion_total_segundos / 60
# Convertimos la duración de minutos a horas dividiendo nuevamente entre 60.
duracion_total_horas = duracion_total_minutos/60

# Imprimimos en consola el resumen general del corpus.
print("Locutores:", cantidad_locutores)
print("Grabaciones:", cantidad_grabaciones)
# Usamos f-strings para formatear la salida mostrando solo dos decimales (.2f).
print(f"Duración total en minutos: {duracion_total_minutos:.2f}")
print(f"Duración total en horas: {duracion_total_horas:.2f}")

# El método .describe() de pandas genera automáticamente estadísticas descriptivas 
# de una columna numérica (conteo, media, desviación estándar, mínimo, máximo y cuartiles).
estadisticas_duracion = inventario_audio["duracion_segundos"].describe()

print("\nEstadísticas de duración por grabación, en segundos:")
# Como .describe() devuelve un objeto de tipo Series, usamos .to_frame() para convertirlo 
# en un DataFrame de una sola columna (nombrada "duracion_segundos").
# Luego, usamos display() para mostrarlo con el formato de tabla estilizada del notebook.
display(estadisticas_duracion.to_frame(name="duracion_segundos"))

# Definimos una lista con los nombres de las columnas que contienen la información técnica del formato.
columnas_formato = [
    "frecuencia_muestreo",
    "canales",
    "formato",
    "subtipo"
]

# Agrupamos los datos del inventario basándonos en estas 4 columnas técnicas.
# .groupby() junta todas las filas que tengan exactamente la misma combinación de estos 4 valores.
grupos_formato = inventario_audio.groupby(columnas_formato)

# Usamos .size() sobre los grupos creados para contar cuántas grabaciones (filas) 
# existen dentro de cada grupo (es decir, cuántos audios comparten el mismo formato).
# Esto devuelve una Serie de pandas con un índice múltiple.
resumen_formatos = grupos_formato.size()

# .reset_index() convierte el índice múltiple nuevamente en columnas normales de un DataFrame.
# El parámetro name="grabaciones" asigna ese nombre a la nueva columna que contiene los conteos 
# calculados por .size().
resumen_formatos = resumen_formatos.reset_index(name = "grabaciones")

print("\nFormatos presentes en el corpus:")
# Mostramos la tabla resumen final donde se ven las combinaciones de formatos y cuántos audios hay de cada una.
display(resumen_formatos)

# 8. Resumir las grabaciones y la duración por locutor

In [ ]:
# Agrupa los datos del inventario ('DataFrame') por la columna "locutor"
grupos_locutor = inventario_audio.groupby("locutor")

# Aplica múltiples operaciones matemáticas (.agg) a diferentes columnas para cada locutor agrupado
resumen_por_locutor = grupos_locutor.agg(
    grabaciones = ("archivo", "count"),                 # Cuenta cuántos archivos de audio tiene
    capitulos = ("capitulo", "nunique"),                # Cuenta cuántos capítulos únicos ha leído
    duracion_total_segundos = ("duracion_segundos", "sum"), # Suma la duración de todos sus audios
    duracion_media_segundos = ("duracion_segundos", "mean"),# Saca el promedio de duración por audio
    duracion_minima_segundos = ("duracion_segundos", "min"),# Encuentra su audio más corto
    duracion_maxima_segundos = ("duracion_segundos", "max") # Encuentra su audio más largo
)

# Convierte el índice (que tras el groupby era 'locutor') de nuevo en una columna normal
resumen_por_locutor = resumen_por_locutor.reset_index()

# Crea una nueva columna convirtiendo los segundos totales a minutos para mejor lectura
resumen_por_locutor["duracion_total_minutos"] = (
    resumen_por_locutor["duracion_total_segundos"] / 60)

# Ordena la tabla resultante basándose en la cantidad de grabaciones (de menor a mayor)
resumen_por_locutor = resumen_por_locutor.sort_values(
    by = "grabaciones",
    ascending = True)

# Define una lista con las columnas específicas que queremos mantener en nuestra tabla final
columnas_resumen = [
    "locutor",
    "grabaciones",
    "capitulos",
    "duracion_total_minutos",
    "duracion_media_segundos"]

# Filtra el DataFrame original para quedarse solo con esas columnas limpias
tabla_resumen = resumen_por_locutor[columnas_resumen]

# Muestra la tabla en pantalla, redondeando todos los números decimales a 2 posiciones
display(tabla_resumen.round(2))

# --- Estadísticas generales (de la segunda imagen) ---
# Encuentra el valor mínimo y máximo de grabaciones entre todos los locutores
minimo_grabaciones = resumen_por_locutor["grabaciones"].min()
maximo_grabaciones = resumen_por_locutor["grabaciones"].max()

# Encuentra el mínimo y máximo de minutos totales producidos por los locutores
minimo_minutos = resumen_por_locutor["duracion_total_minutos"].min()
maximo_minutos = resumen_por_locutor["duracion_total_minutos"].max()

# Encuentra la menor y mayor cantidad de capítulos leídos por un solo locutor
minimo_capitulos = resumen_por_locutor["capitulos"].min()
maximo_capitulos = resumen_por_locutor["capitulos"].max()

# Imprime estos resultados estadísticos en la consola
print("Mínimo de grabaciones por locutor:", minimo_grabaciones)
print("Máximo de grabaciones por locutor:", maximo_grabaciones)

# Imprime los minutos con formato de 2 decimales (.2f)
print(f"Mínimo de minutos por locutor: {minimo_minutos:.2f}")
print(f"Máximo de minutos por locutor: {maximo_minutos:.2f}")

print("Mínimo de capítulos por locutor:", minimo_capitulos)
print("Máximo de capítulos por locutor:", maximo_capitulos)

# 9. Graficar las grabaciones y la duración por locutor

In [ ]:
# Importa la librería estándar para crear gráficos científicos en Python
import matplotlib.pyplot as plt

# Prepara los datos para graficar, asegurándose de que estén ordenados por 'grabaciones'
tabla_grafica = resumen_por_locutor.sort_values(
    by = "grabaciones",
    ascending = True)

# Extrae las columnas como listas nativas de Python (.tolist()) para facilitar el graficado
identificadores = tabla_grafica["locutor"].tolist()
cantidades = tabla_grafica["grabaciones"].tolist()
duraciones = tabla_grafica["duracion_total_minutos"].tolist()

# Crea un lienzo (figura) con 2 subgráficas (1 fila, 2 columnas)
figura, ejes = plt.subplots(
    nrows = 1,
    ncols = 2,
    figsize = (14, 12), # Define un lienzo grande (14 de ancho, 12 de alto)
    sharey = True)      # Ambas gráficas compartirán el eje Y (los nombres de locutores)

# --- Gráfica 1 (Izquierda): Cantidad de grabaciones ---
# Dibuja un gráfico de barras horizontales (.barh) en el primer eje (ejes[0])
ejes[0].barh(
    identificadores, # Eje Y: IDs de los locutores
    cantidades,      # Eje X: Número de grabaciones
    color = "steelblue") # Color azul acero

# Configura título, etiquetas y la cuadrícula visual
ejes[0].set_title("Grabaciones por locutor")
ejes[0].set_xlabel("Número de grabaciones")
ejes[0].set_ylabel("Identificador del locutor")
ejes[0].set_xlim(left = 0) # Fuerza que el eje X empiece en 0
ejes[0].grid(axis = "x", alpha = 0.3) # Cuadrícula vertical suave (30% opacidad)
ejes[0].set_axisbelow(True) # Manda la cuadrícula al fondo, detrás de las barras coloreadas

# --- Gráfica 2 (Derecha): Duración total en minutos ---
# Dibuja el segundo gráfico de barras horizontales en el segundo eje (ejes[1])
ejes[1].barh(
    identificadores,
    duraciones,
    color = "seagreen") # Color verde mar

ejes[1].set_title("Duración por locutor")
ejes[1].set_xlabel("Duración total en minutos")
ejes[1].set_xlim(left = 0)
ejes[1].grid(axis = "x", alpha = 0.3)
ejes[1].set_axisbelow(True)

# Coloca un súper título general para abarcar toda la figura
figura.suptitle(
    "Distribución de datos entre los 40 locutores",
    fontsize = 16)

# Ajusta automáticamente los márgenes para que los textos y títulos no se sobrepongan
figura.tight_layout()

# Renderiza y muestra la figura final
plt.show()

# 10. Analizar la distribución de duración de los audios

In [ ]:
# Extrae solo la columna de duraciones en segundos de todos los audios individuales del corpus
duraciones_audio = inventario_audio["duracion_segundos"]

# Crea un nuevo lienzo con 2 gráficas (esta vez 2 filas y 1 columna)
figura, ejes = plt.subplots(
    nrows = 2,
    ncols = 1,
    figsize = (12, 7),
    gridspec_kw = {"height_ratios": [3, 1]}, # Proporción: La gráfica superior será 3 veces más alta que la inferior
    sharex = True) # Compartirán el eje X (duración en segundos)

# --- Gráfica 1 (Arriba): Histograma ---
# Crea un histograma para visualizar estadísticamente cómo se agrupan las duraciones
ejes[0].hist(
    duraciones_audio,
    bins = 30,             # Divide todos los datos en 30 "cubetas" o barras
    color = "steelblue",
    edgecolor = "white")   # Borde blanco para delimitar y separar visualmente las barras

ejes[0].set_title("Distribución de duración de las grabaciones")
ejes[0].set_ylabel("Número de grabaciones")
ejes[0].grid(axis = "y", alpha = 0.3)
ejes[0].set_axisbelow(True)

# --- Gráfica 2 (Abajo): Diagrama de caja (Boxplot) ---
# Dibuja un boxplot para ver la dispersión, la mediana y los valores atípicos (outliers)
ejes[1].boxplot(
    duraciones_audio,
    vert = False)          # Dibuja la caja horizontalmente

ejes[1].set_xlabel("Duración en segundos")
ejes[1].set_yticks([])     # Oculta los números del eje Y del boxplot (solo confunden visualmente aquí)
ejes[1].set_xlim(left = 0)

figura.tight_layout()
plt.show()

# --- Cálculos estadísticos detallados ---
duracion_minima = duraciones_audio.min()
duracion_maxima = duraciones_audio.max()
duracion_media = duraciones_audio.mean()
duracion_mediana = duraciones_audio.median()

# Imprime los resultados
print(f"Duración mínima: {duracion_minima:.2f} segundos")
print(f"Duración máxima: {duracion_maxima:.2f} segundos")
print(f"Duración media: {duracion_media:.2f} segundos")
print(f"Duración mediana: {duracion_mediana:.2f} segundos")

# 11. Comprobar la uniformidad del formato de audio

In [ ]:
print("Formatos presentes en el corpus:")
# Imprime el DataFrame de resumen de formatos (calculado previamente) ocultando su índice interno
print(resumen_formatos.to_string(index = False))

# Extrae los valores únicos para ver si hay audios con distintas calidades mezcladas
frecuencias = inventario_audio["frecuencia_muestreo"].unique()
canales = inventario_audio["canales"].unique()

# Si la longitud del arreglo de frecuencias es 1, significa que TODOS los audios comparten exactamente la misma frecuencia
frecuencia_uniforme = len(frecuencias) == 1
# Verifica la misma lógica para los canales de audio (ej. que todos sean mono)
canales_uniformes = len(canales) == 1

print("\nFrecuencia de muestreo uniforme:", frecuencia_uniforme)
print("Número de canales uniforme:", canales_uniformes)

# Si pasó la prueba de uniformidad, imprime cuál es ese formato oficial (leyendo la posición [0] del arreglo)
if frecuencia_uniforme:
    print("Frecuencia de muestreo, en Hz:", frecuencias[0])

if canales_uniformes:
    print("Canales por grabación:", canales[0])

# --- Verificación de audios corruptos o erróneos ---
# Crea "máscaras" booleanas: detecta filas cuyo número de muestras o duración es cero o menor
audios_sin_muestras = inventario_audio["numero_muestras"] <= 0
audios_sin_duracion = inventario_audio["duracion_segundos"] <= 0

# Filtra el inventario buscando audios que cumplan con la primera O (|) con la segunda condición de error
registros_invalidos = inventario_audio[
    audios_sin_muestras | audios_sin_duracion]

print("Registros con muestras o duración no válidas:", len(registros_invalidos))

# Si encontró registros corruptos (el DataFrame 'no' está vacío), los despliega para analizarlos
if not registros_invalidos.empty:
    display(registros_invalidos)

# 12. Comprobar la lectura y amplitud de los audios

In [ ]:
import numpy as np

# Se crean listas vacías para guardar qué audios pasaron la prueba y cuáles arrojaron error al leerlos
registros_calidad = []
errores_decodificacion = []

# Define un umbral para detectar saturación (clipping). 0.999 es casi el volumen máximo posible en el formato digital (1.0)
umbral_amplitud = 0.999

# Itera sobre cada fila del inventario. 'itertuples' es mucho más eficiente/rápido que 'iterrows' en Pandas
for indice, registro in enumerate(
    inventario_audio.itertuples(index = False),
    start = 1):
    try:
        # Intenta decodificar/abrir el archivo físico usando la ruta registrada
        # Forzamos dtype="float32" para estandarizar la lectura de los datos sin importar el formato original
        senal, frecuencia = sf.read(
            registro.ruta,
            dtype = "float32")

        # 1. Verifica que la lectura no haya devuelto una señal vacía
        if senal.size == 0:
            raise ValueError("La señal está vacía.")

        # 2. Verifica que no existan valores matemáticamente infinitos o nulos (NaN) en la onda sonora
        if not np.isfinite(senal).all():
            raise ValueError("La señal contiene valores no finitos.")

        # 3. Comprueba que la frecuencia de muestreo del archivo real coincida con lo que se reportó en el inventario (metadatos)
        if frecuencia != registro.frecuencia_muestreo:
            raise ValueError("La frecuencia no coincide con el inventario.")

        # 4. Comprueba que la cantidad total de puntos de datos coincida con la longitud registrada
        if len(senal) != registro.numero_muestras:
            raise ValueError("El número de muestras no coincide.")
        # --- Análisis de Amplitud ---
        # Convierte todos los valores de la onda a positivos (valor absoluto) para buscar el pico máximo
        amplitud_absoluta = np.abs(senal)
        # Extrae el valor más alto encontrado en todo el audio y lo convierte a float estándar de Python
        amplitud_maxima = float(amplitud_absoluta.max())

        # Calcula la energía RMS (Root Mean Square - Raíz Cuadrada Media)
        # 1. Eleva toda la señal al cuadrado (forzando precisión float64 para evitar desbordamientos de memoria)
        cuadrados = np.square(senal, dtype = np.float64)
        # 2. Saca el promedio de esos cuadrados y luego su raíz cuadrada
        amplitud_rms = float(np.sqrt(cuadrados.mean()))

        # --- Detección de saturación (clipping) ---
        # Crea un arreglo booleano (True/False) indicando qué muestras superan o igualan el umbral (0.999)
        muestras_al_limite = amplitud_absoluta >= umbral_amplitud
        # Calcula qué porcentaje del audio total está "saturado" o rozando el límite digital
        porcentaje_al_limite = float(muestras_al_limite.mean() * 100)

        # Si el audio pasó todas las pruebas y se calculó su amplitud, guarda sus datos en un diccionario
        resultado = {
            "archivo": registro.archivo,
            "locutor": registro.locutor,
            "amplitud_maxima": amplitud_maxima,
            "rms": amplitud_rms,
            "porcentaje_al_limite": porcentaje_al_limite,
            "senal_nula": amplitud_maxima == 0} # True si el audio es puro silencio estático (amplitud 0 en todos lados)

        # Añade este diccionario a la lista de audios exitosos
        registros_calidad.append(resultado)

    # Si 'sf.read()' falló al abrir el archivo, o si se disparó algún 'raise ValueError' nuestro, se captura aquí
    except (OSError, RuntimeError, ValueError) as error:
        # Crea un registro documentando exactamente qué archivo falló y el motivo técnico
        incidencia = {
            "archivo": registro.archivo,
            "error": str(error)}
        
        # Añade este error a la lista de fallos
        errores_decodificacion.append(incidencia)

    # Imprime un indicador de progreso cada 500 audios procesados para que el usuario sepa que el código no se ha congelado
    if indice % 500 == 0:
        print("Grabaciones revisadas:", indice)

# --- Resumen de calidad --- (de la foto 30)
# Convierte las listas de resultados en DataFrames de Pandas para facilitar su manejo
calidad_audio = pd.DataFrame(registros_calidad)

tabla_errores_decodificacion = pd.DataFrame(
    errores_decodificacion,
    columns=["archivo", "error"])

# Imprime los conteos finales de éxito y fracaso
print("\nAudios leídos correctamente:", len(calidad_audio))
print("Errores:", len(tabla_errores_decodificacion))

# Si la tabla de calidad está vacía, significa que NINGÚN audio sirvió. Aborta el programa.
if calidad_audio.empty:
    raise ValueError("No se pudo analizar ninguna señal.")

# Suma cuántos audios resultaron ser puro silencio (True equivale a 1)
cantidad_nulos = int(calidad_audio["senal_nula"].sum())

# Filtra la tabla para buscar audios que tengan al menos > 0% de saturación
audios_al_limite = calidad_audio["porcentaje_al_limite"] > 0
cantidad_al_limite = int(audios_al_limite.sum())

print("Audios con señal completamente nula:", cantidad_nulos)
print("Audios con muestras cercanas al límite:", cantidad_al_limite)

# Si hubo errores de decodificación físicos, muestra la tabla completa para que el analista los repare
if not tabla_errores_decodificacion.empty:
    print(tabla_errores_decodificacion.to_string(index = False))

# 13. Identificar los audios cercanos al límite digital

In [ ]:
# Crea una máscara para aislar los audios que presentaron saturación (clipping) mayor a 0
condicion_limite = calidad_audio["porcentaje_al_limite"] > 0
# Crea un nuevo DataFrame independiente (.copy()) solo con esos audios problemáticos
audios_para_revision = calidad_audio.loc[condicion_limite].copy()

# Selecciona columnas clave del inventario original (metadatos) que queremos anexar a nuestra tabla de revisión
columnas_inventario = ["archivo", "ruta", "duracion_segundos"]

# Fusiona (merge) la tabla de revisión con la tabla original basándose en el nombre del archivo
audios_para_revision = audios_para_revision.merge(
    inventario_audio[columnas_inventario],
    on = "archivo",          # Llave de cruce
    how = "left",            # Mantiene todos los registros de la tabla izquierda (la de revisión)
    validate = "one_to_one") # Se asegura matemáticamente de que no se dupliquen registros por accidente

# Ordena los audios de mayor a menor gravedad de saturación (descendente)
audios_para_revision = audios_para_revision.sort_values(
    by = "porcentaje_al_limite",
    ascending = False)

# Define el orden exacto de las columnas para presentar el reporte
columnas_revision = [
    "archivo",
    "locutor",
    "duracion_segundos",
    "amplitud_maxima",
    "rms",
    "porcentaje_al_limite"]

print("Audios que requieren inspección:", len(audios_para_revision))

# Imprime la tabla final formateando los números flotantes a 6 decimales de precisión
print(audios_para_revision[columnas_revision].to_string(
    index = False,
    float_format = "{:.6f}".format))

# 14. Visualizar y escuchar los audios señalados

In [ ]:
# Importa la herramienta de IPython/Jupyter para incrustar un reproductor multimedia directamente en el cuaderno
from IPython.display import Audio

# Inicia un bucle para graficar y escuchar CADA UNO de los audios marcados para revisión
for registro in audios_para_revision.itertuples(index = False):
    
    # Lee físicamente el archivo problemático otra vez
    senal, frecuencia = sf.read(registro.ruta, dtype = "float32")
    
    # Crea un eje X de tiempo real en segundos (índice de muestra / frecuencia de muestreo)
    tiempos = np.arange(len(senal)) / frecuencia
    
    # Busca el punto exacto (índice) donde la onda es más violenta (clipping)
    indice_maximo = int(np.argmax(np.abs(senal)))
    tiempo_maximo = indice_maximo / frecuencia
    
    # Define un margen de "zoom" equivalente a 10 milisegundos (0.010 segundos * frecuencia de muestreo)
    margen_muestras = int(0.010 * frecuencia)
    
    # Calcula dónde empieza y dónde termina el acercamiento (zoom), asegurándose de no salirse de los bordes del audio (min/max)
    inicio = max(0, indice_maximo - margen_muestras)
    fin = min(len(senal), indice_maximo + margen_muestras + 1)
    
    # Prepara el lienzo con 2 gráficas apiladas
    figura, ejes = plt.subplots(nrows = 2, ncols = 1, figsize = (12, 6))
    
    # --- Gráfica 1 (Arriba): Onda Completa ---
    ejes[0].plot(tiempos, senal, linewidth = 0.6) # Dibuja todo el audio con una línea delgada
    ejes[0].set_title(registro.archivo)
    ejes[0].set_xlabel("Tiempo en segundos")
    ejes[0].set_ylabel("Amplitud")
    ejes[0].set_ylim(-1.05, 1.05) # Fija los topes del eje Y un poco más allá del -1 y 1 digital
    
    # --- Gráfica 2 (Abajo): Zoom detallado en el punto de saturación --- (de la foto 33)
    ejes[1].plot(
        tiempos[inicio:fin], # Recorta el eje X usando los índices de inicio/fin calculados
        senal[inicio:fin],   # Recorta el eje Y (la onda)
        marker = ".",        # Pone un punto explícito en cada muestra capturada por el micrófono
        linewidth = 0.8)
    
    # Dibuja una línea punteada horizontal roja marcando el umbral de peligro superior (ej. 0.999)
    ejes[1].axhline(
        umbral_amplitud,
        color = "red",
        linestyle = "--")
    
    # Dibuja la línea punteada horizontal roja para el umbral negativo
    ejes[1].axhline(
        -umbral_amplitud,
        color = "red",
        linestyle = "--")
    
    ejes[1].set_title("Acercamiento alrededor del máximo")
    ejes[1].set_xlabel("Tiempo en segundos")
    ejes[1].set_ylabel("Amplitud")
    ejes[1].set_ylim(-1.05, 1.05)
    
    figura.tight_layout()
    plt.show() # Muestra las gráficas para este audio específico
    
    print(f"Máximo de amplitud en: {tiempo_maximo:.4f} segundos")
    
    # Despliega un widget reproductor de audio dentro del Notebook.
    # normalize=False evita que IPython le baje el volumen automáticamente, permitiendo al analista escuchar el "clipping" real.
    display(Audio(data = senal, rate = frecuencia, normalize = False))

# 15. Escuchar el entorno del pico del audio con chasquido

In [ ]:
# Definimos el nombre del archivo de audio específico que queremos analizar
nombre_audio_revision ="6295-244435-0010.flac"

# Filtramos el DataFrame 'inventario_audio' para obtener la fila que coincide con el nombre del archivo
fila_audio = inventario_audio.loc[
    inventario_audio["archivo"] == nombre_audio_revision
]

# Extraemos la cadena de texto con la ruta del archivo desde la primera coincidencia (índice 0)
ruta_audio_revision = fila_audio.iloc[0]["ruta"]

# Cargamos el archivo de audio 
# Esto devuelve la señal de audio (array) y su frecuencia de muestreo.
senal_revision, frecuencia_revision = sf.read(
    ruta_audio_revision,
    dtype="float32"
)

# Buscamos el índice dentro del array donde la amplitud absoluta es máxima (el volumen más alto)
indice_pico= int(np.argmax(np.abs(senal_revision)))
# Convertimos ese índice (muestras) a tiempo en segundos dividiendo por la frecuencia de muestreo
tiempo_pico= indice_pico / frecuencia_revision

# Calculamos un margen en muestras (equivalente a 0.5 segundos).
marge = int(0.5 * frecuencia_revision)

# Definimos el índice de inicio del fragmento, usando max() para asegurar que no sea un número negativo
inicio_fragmento = max(0,indice_pico-marge)
# Definimos el índice de fin, usando min() para asegurar que no sobrepase la longitud total del audio
fin_fragmento = min(
    len(senal_revision),
    indice_pico+marge+1)

# Hacemos un "slicing" (recorte) del array original para quedarnos solo con el pedazo de interés
fragmento_revision = senal_revision[
    inicio_fragmento:fin_fragmento]

# Imprimimos en consola los resultados calculados, formateando los tiempos a 4 decimales
print("Archivo: ", nombre_audio_revision)
print(f"Tiempo pico:  {tiempo_pico: .4f} segundos")
print(f"Inicio fragmento: {inicio_fragmento/frecuencia_revision:.4f} s")
print(f"Fin fragmento: {fin_fragmento/frecuencia_revision:.4f} s")

# Renderizamos un reproductor de audio interactivo
display(Audio(
    data = fragmento_revision,
    rate = frecuencia_revision,
    normalize = False))

# 16. Registrar la incidencia de chasquido audible

In [ ]:
# Creamos un diccionario para almacenar los metadatos y observaciones de una anomalía detectada en el audio
registro_incidencia = {
    "archivo": nombre_audio_revision,
    "locutor": fila_audio.iloc[0]["locutor"],
    "incidencia": "Chasquido audible",
    "observacion": (
        "Se escucha en el fragmento alrededor del máximo "
        "de amplitud. Causa no determinada."),
    "decision":"Conservar para el experimento inicial"}

# Convertimos el diccionario en un DataFrame de pandas (pd) pasándolo dentro de una lista.
incidencias_manuales = pd.DataFrame([registro_incidencia])

# Hacemos una copia independiente del DataFrame original 'inventario_audio' 
# para trabajar sobre ella sin modificar los datos de origen.
inventario_modelado = inventario_audio.copy()

# Imprimimos el DataFrame de incidencias en formato texto, ocultando el índice numérico automático de pandas
print(incidencias_manuales.to_string(index=False))

# Imprimimos el número total de grabaciones que tiene el inventario copiado (filas del DataFrame).
print("\nGrabaciones para el modelo:", len(inventario_modelado))

# Imprimimos la cantidad de locutores únicos que existen en el DataFrame.
print("Locutores:",inventario_modelado["locutor"].nunique())

# 17. Separar entrenamiento,validación y prueba

In [ ]:
# Definimos una semilla (seed) fija para garantizar que la aleatoriedad sea reproducible
semilla = 7
# Inicializamos el generador de números aleatorios de NumPy con la semilla proporcionada
generador = np.random.default_rng(semilla)

# Ordenamos el DataFrame por 'locutor' y luego por 'archivo', creando una copia.
datos_particionados = inventario_modelado.sort_values(
    by= ["locutor", "archivo"]).copy()

# Reiniciamos el índice del DataFrame (descartando el anterior) para que sea continuo desde 0
datos_particionados = datos_particionados.reset_index(drop = True)

# Creamos una nueva columna vacía llamada "particion" para etiquetar posteriormente los datos
datos_particionados["particion"] = ""

# Agrupamos por locutor e iteramos para particionar los datos por cada persona
for locutor, grupo in datos_particionados.groupby("locutor"):

  # Extraemos los índices de las filas del locutor actual y los pasamos a un arreglo de NumPy
  indices = grupo.index.to_numpy()

  # Mezclamos aleatoriamente los índices de este locutor
  indices_mezclados = generador.permutation(indices)

  # Determinamos la cantidad total de grabaciones que tiene este locutor
  cantidad = len(indices_mezclados)

  # Calculamos el 20% del total para el conjunto de validación, redondeado a número entero
  cantidad_validacion = int(round(cantidad*0.20))

  # Calculamos otro 20% del total para el conjunto de prueba, redondeado a entero
  cantidad_prueba = int(round(cantidad*0.20))

  # El resto (aproximadamente un 60%) se destina al conjunto de entrenamiento
  cantidad_entrenamiento = (
      cantidad - cantidad_validacion - cantidad_prueba)
  
  # Definimos el límite donde termina la sección de entrenamiento en el arreglo
  fin_entrenamiento = cantidad_entrenamiento 

  # Definimos el límite donde termina la sección de validación en el arreglo
  fin_validacion = fin_entrenamiento + cantidad_validacion

  # Extraemos los índices mezclados correspondientes a la porción de entrenamiento
  indices_entrenamiento = indices_mezclados[:fin_entrenamiento]

  # Extraemos los índices mezclados correspondientes a la porción de validación
  indices_validacion = indices_mezclados[
      fin_entrenamiento:fin_validacion]

  # Extraemos los índices mezclados restantes para la porción de prueba
  indices_prueba = indices_mezclados[fin_validacion:]

  # Asignamos la etiqueta "entrenamiento" a las filas correspondientes en la columna "particion"
  datos_particionados.loc[
      indices_entrenamiento, "particion"] = "entrenamiento"

  # Asignamos la etiqueta "validacion" a las filas correspondientes en la columna "particion"
  datos_particionados.loc[
      indices_validacion, "particion"] = "validacion"
      
  # Asignamos la etiqueta "prueba" a las filas correspondientes en la columna "particion"
  datos_particionados.loc[
      indices_prueba, "particion"] = "prueba"
  
# Creamos un DataFrame independiente (copia) filtrando solo los datos de entrenamiento
datos_entrenamiento = datos_particionados.loc[
    datos_particionados["particion"] == "entrenamiento"].copy()

# Creamos un DataFrame independiente (copia) filtrando solo los datos de validación
datos_validacion = datos_particionados.loc[
    datos_particionados["particion"] == "validacion"].copy()
# Creamos un DataFrame independiente (copia) filtrando solo los datos de prueba
datos_prueba = datos_particionados.loc[
    datos_particionados["particion"] == "prueba"].copy()

# Generamos un resumen agrupando por partición y calculando la cantidad de archivos, locutores únicos y duración total
resumen_particiones = datos_particionados.groupby("particion").agg(
    grabaciones = ("archivo", "count"),
    locutores = ("locutor", "nunique"),
    duracion_segundos = ("duracion_segundos", "sum"))

# Transformamos la duración de segundos a minutos dividiendo por 60.
resumen_particiones["duracion_minutos"] = (
    resumen_particiones["duracion_segundos"] / 60)

# Definimos una lista para ordenar y seleccionar específicamente las columnas que queremos mostrar
columnas_particiones = [
    "grabaciones",
    "locutores",
    "duracion_minutos"]

# Imprimimos el resumen mostrando las columnas seleccionadas, redondeando a 2 decimales y convertido a string
print(resumen_particiones[columnas_particiones].round(2).to_string())

# 18.Verificar la separación de los conjuntos de datos

In [ ]:
# Convertimos la columna de rutas de cada subconjunto en un "set" (conjunto matemático).
# Esto elimina duplicados internamente y permite hacer operaciones de conjuntos muy rápidas.
rutas_entrenamiento = set(datos_entrenamiento["ruta"])
rutas_validacion = set(datos_validacion["ruta"])
rutas_prueba = set(datos_prueba["ruta"])

# Usamos el operador de intersección '&' para encontrar si hay rutas compartidas (fugas de datos) 
# entre el conjunto de entrenamiento y el de validación.
coincidencias_entrenamiento_validacion = (
    rutas_entrenamiento & rutas_validacion)

# Buscamos archivos que estén al mismo tiempo en entrenamiento y en prueba.
coincidencias_entrenamiento_prueba = (
    rutas_entrenamiento & rutas_prueba)

# Buscamos archivos compartidos entre validación y prueba.
coincidencias_validacion_prueba = (
    rutas_validacion & rutas_prueba)

# Imprimimos la cantidad de elementos (archivos compartidos) en cada intersección. Lo ideal es que todas sean 0.
print("Coincidencias entre entrenamiento y validacion: ", 
      len(coincidencias_entrenamiento_validacion))
print("Coincidencias entre entrenamiento y prueba: ", 
      len(coincidencias_entrenamiento_prueba))
print("Coincidencias entre validacion y prueba: ", 
      len(coincidencias_validacion_prueba))

# Contamos si existe alguna fila duplicada en la columna "ruta" de todo el dataset particionado.
cantidad_duplicados = int(
    datos_particionados["ruta"].duplicated().sum())

# Si hay al menos un duplicado, lanzamos un error para detener la ejecución.
if cantidad_duplicados > 0:
  raise ValueError("Existen rutas duplicadas en el inventario.")

# Verificamos si alguna de las intersecciones calculadas anteriormente tiene contenido (evalúan a True si no están vacías).
if ( coincidencias_entrenamiento_validacion
    or coincidencias_entrenamiento_prueba
    or coincidencias_validacion_prueba):
  raise ValueError("Hay archivos compartidos entre particiones.")

# Creamos un set con las rutas del inventario original para comparar.
rutas_originales = set(inventario_modelado["ruta"])

# Unimos los tres sets de las particiones usando el operador de unión '|' para reconstruir el dataset completo.
rutas_particionadas =(
    rutas_entrenamiento | rutas_validacion | rutas_prueba)

# Comparamos si la unión de las particiones es exactamente igual al inventario original.
if rutas_particionadas != rutas_originales:
  raise ValueError("Las particiones no conservan todos los archivos.")

# Extraemos todos los locutores únicos del inventario original para saber cuántos y cuáles deberían ser.
locutores_esperados= set(inventario_modelado["locutor"])

# Agrupamos los tres DataFrames particionados en un diccionario para iterar sobre ellos fácilmente.
conjuntos = {
    "entrenamiento": datos_entrenamiento,
    "validacion": datos_validacion,
    "prueba": datos_prueba}

# Iteramos sobre el diccionario obteniendo el nombre del conjunto y su respectivo DataFrame.
for nombre,conjunto in conjuntos.items():

  # Extraemos los locutores únicos presentes en el conjunto actual
  locutores_presentes = set(conjunto["locutor"])

  # Verificamos si los locutores en esta partición son exactamente los mismos que en el inventario original.
  if locutores_presentes != locutores_esperados:
    
    # Si falta o sobra alguno, lanzamos un error indicando en qué partición ocurrió.
    raise ValueError(
        f"faltan locutores en la partición: {nombre}")
    
# Si el código llega hasta aquí sin lanzar ningún error, se imprimen los mensajes de éxito.
print("\nVerificación correcta.")
print("Cada grabación pertenece a una sola partición")
print("Todos los locutores están presentes en los tres conjuntos.")

# 19. Definir los parámetros de extracción de MFCC

In [ ]:
# Importamos la librería librosa, una de las más utilizadas en Python para análisis de audio y extracción de características
import librosa

# Definimos la frecuencia de muestreo objetivo para procesar los audios (10,000 Hz o 10 kHz)
frecuencia_objetivo = 10000

# Definimos la duración temporal de cada ventana de análisis en milisegundos
duracion_ventana_ms=25

# Definimos el desplazamiento temporal (hop length) entre ventanas consecutivas en milisegundos
desplazamiento_ms= 10

# Establecemos el número de coeficientes cepstrales en las frecuencias mel (MFCC) a calcular por trama
numero_mfcc = 20

# Establecemos el número de filtros o bandas Mel a utilizar en el banco de filtros
numero_filtros_mel = 40

# Definimos el número de puntos para la Transformada Rápida de Fourier (FFT), que afecta la resolución frecuencial
numero_fft =512

# Nota experta: Aquí hay un error tipográfico en el nombre de la variable ('frecuencia_minimo' en vez de 'frecuencia_minima').
frecuencia_minima= 20

# Calculamos la frecuencia máxima analizable, que según el teorema de Nyquist es exactamente la mitad de la frecuencia de muestreo
frecuencia_maxima = frecuencia_objetivo / 2

# Convertimos la duración de la ventana de milisegundos a cantidad de muestras continuas (samples)
longitud_ventana = int(
    frecuencia_objetivo * duracion_ventana_ms / 1000)

# Convertimos el tiempo de desplazamiento de milisegundos a cantidad de muestras (samples de salto)
salto_ventana = int(
    frecuencia_objetivo * desplazamiento_ms / 1000)

# Validamos matemáticamente que la ventana quepa dentro del tamaño de la FFT.
# Si la ventana es mayor, se perdería información (truncamiento).
if numero_fft < longitud_ventana:
  raise ValueError(
      "El tamaño de la FFT debe ser igual o mayor que la ventana."
  )

# Validamos que no se intente extraer un número de coeficientes MFCC mayor al número de bandas Mel de origen.
if numero_mfcc > numero_filtros_mel:
  raise ValueError(
      "Los MFCC no deben superar el número de filtros Mel."
  )

# Imprimimos todos los parámetros calculados para confirmar la configuración antes de extraer características.
print("Frecuencia de muestreo:", frecuencia_objetivo, "Hz")
print("Longitud de ventana:", longitud_ventana, "muestras")
print("Desplazamiento:", salto_ventana, "muestras")
print("Tamaño de FFT:", numero_fft)
print("Filtros Mel:", numero_filtros_mel)
print("MFCC por trama:", numero_mfcc)
print("Versión de librosa:", librosa.__version__)

# 20. Definir la función de extracción de MFCC


In [ ]:
# Definimos una función que encapsula todo el proceso de extracción de características (MFCC) de un archivo de audio
def extraer_mfcc(ruta_audio):

  # Leemos el archivo de audio asegurando que la ruta sea un string y forzando el formato a flotante de 32 bits
  senal,frecuencia = sf.read(str(ruta_audio), dtype ="float32")

  # Validamos que la frecuencia de muestreo del audio coincida con la que configuramos globalmente
  if frecuencia != frecuencia_objetivo:
    raise ValueError(
        f"Frecuencia de muestreo inesperada: {frecuencia} Hz.")
    
  # Comprobamos que el audio tenga un solo canal (mono).
  if senal.ndim != 1:
    raise ValueError("Se esperaba una grabación mono.")
  
  # Validamos que el audio tenga al menos el tamaño de una ventana de FFT, de lo contrario no se puede procesar
  if len(senal)< numero_fft:
    raise ValueError("La grabación es demasiado corta.")
  
  # Nos aseguramos de que la señal no contenga valores infinitos o "Not a Number" (NaN) que rompan el modelo
  if not np.isfinite(senal).all():
    raise ValueError("Se encontraron valores no finitos.")

  # Calculamos el espectrograma Mel, que representa la energía de la señal en diferentes bandas de frecuencia (escala Mel) a lo largo del tiempo
  espectrograma_mel = librosa.feature.melspectrogram(
      y=senal,
      sr=frecuencia,
      n_fft=numero_fft,
      hop_length=salto_ventana,
      win_length=longitud_ventana,
      window = "hann",
      center = False,
      power = 2.0,
      n_mels = numero_filtros_mel,
      fmin = frecuencia_minima,
      fmax = frecuencia_maxima)
  
  # Convertimos la escala de potencia (amplitud al cuadrado) a decibelios (escala logarítmica), 
  # lo cual simula mejor cómo percibe el volumen el oído humano
  mel_logaritmico = librosa.power_to_db(
      espectrograma_mel,
      ref = 1.0,
      amin = 1e-10,
      top_db = None)
  
  # Extraemos los Coeficientes Cepstrales en las Frecuencias Mel (MFCC) a partir del espectrograma logarítmico
  coeficientes = librosa.feature.mfcc(
      S=mel_logaritmico,
      n_mfcc=numero_mfcc,
      dct_type= 2,
      norm = "ortho",
      lifter = 0)
  
  # Transponemos la matriz (.T) para que las filas sean el tiempo (tramas) y las columnas sean los coeficientes (características),
  # y hacemos una copia independiente en memoria
  matriz_mfcc = coeficientes.T.copy()

  # Hacemos una última validación de seguridad para garantizar que la matriz resultante no tenga valores corruptos (infinitos o NaN)
  if not np.isfinite(matriz_mfcc).all():
    raise ValueError("Los MFCC contienen valores no finitos")
  
  # Retornamos la matriz con las características extraídas, lista para usarse en Machine Learning
  return matriz_mfcc

# 21. Comprobar los MFCC e una grabación de entrenamiento

In [ ]:

# Seleccionamos la primera fila (índice 0) del DataFrame de entrenamiento para usarla como ejemplo
audio_ejemplo = datos_entrenamiento.iloc[0]

# Llamamos a la función que definimos en el paso anterior para extraer las características del audio seleccionado
mfcc_ejemplo = extraer_mfcc(audio_ejemplo["ruta"])

# Obtenemos la cantidad de tramas (ventanas de tiempo) leyendo la primera dimensión de la matriz resultante
cantidad_tramas = mfcc_ejemplo.shape[0]

# Obtenemos la cantidad de coeficientes (MFCCs) leyendo la segunda dimensión de la matriz
cantidad_coeficientes = mfcc_ejemplo.shape[1]

# Calculamos matemáticamente cuántas tramas debería haber generado la extracción.
tramas_esperadas = (
    1 + (audio_ejemplo["numero_muestras"]- numero_fft)
    // salto_ventana)

# Validamos que el número de tramas extraídas coincida con el cálculo teórico
if cantidad_tramas != tramas_esperadas:
  raise ValueError("El número de tramas no coincide con lo esperado.")

# Validamos que el número de coeficientes coincida con el parámetro de configuración (numero_mfcc)
if cantidad_coeficientes != numero_mfcc:
  raise ValueError("El número de coeficientes no coincide.")

# Imprimimos los metadatos básicos del archivo procesado
print("Archivo: ", audio_ejemplo["archivo"])
print("Locutor: ", audio_ejemplo["locutor"])

# Imprimimos la duración.
print(f"Duración: {audio_ejemplo['duracion_segundos']:.2f} segundos")

# Imprimimos las dimensiones de la matriz generada
print("Dimensiones de la matriz:", mfcc_ejemplo.shape)
print("Número de tramas:", cantidad_tramas)
print("Coeficientes por tramas:", cantidad_coeficientes)

# Creamos una lista vacía para almacenar los nombres de las columnas de nuestro DataFrame de visualización
nombres_coeficientes = []

# Llenamos la lista con nombres dinámicos (MFCC_0, MFCC_1, etc.) iterando hasta el número máximo de coeficientes
for indice in range(numero_mfcc):
  nombres_coeficientes.append(f"MFCC_{indice}")

# Creamos un DataFrame de pandas tomando solo las primeras 5 tramas (filas) de la matriz para mostrarlas como tabla
tabla_mfcc_ejemplo = pd.DataFrame(
    mfcc_ejemplo[:5],
    columns = nombres_coeficientes)

# Renderizamos la tabla en pantalla redondeando los valores a 2 decimales para facilitar la lectura
display(tabla_mfcc_ejemplo.round(2))

# Inicializamos una figura de matplotlib para graficar los MFCC. Definimos el tamaño a 12 de ancho por 5 de alto.
figura, eje = plt.subplots(figsize = (12,5))

# Creamos un mapa de calor (heatmap) con los datos transpuestos (.T) para que el tiempo quede en el eje X y los MFCC en el Y
imagen = eje.imshow(
    mfcc_ejemplo.T,
    origin = "lower",
    aspect = "auto",
    interpolation = "nearest",
    cmap= "viridis")

# Añadimos un título al gráfico.
eje.set_title("MFCC de una grabación de entrenamiento")

# Etiquetamos el eje X indicando que representa el avance del tiempo (tramas)
eje.set_xlabel("Ìndice de trama")

# Etiquetamos el eje Y indicando que representa los diferentes coeficientes
eje.set_ylabel("Ìndice del coeficiente MFCC")

# Colocamos las marcas en el eje Y correspondientes a cada coeficiente
eje.set_yticks(range(numero_mfcc))

# Reemplazamos los números del eje Y por las etiquetas de texto que creamos antes ("MFCC_0", etc.)
eje.set_yticklabels(nombres_coeficientes)

# Añadimos una barra lateral que sirve como leyenda para interpretar la escala de colores (intensidad)
barra_color = figura.colorbar(imagen, ax =eje)

# Le ponemos un título a la barra de color
barra_color.set_label("Valor del coeficiente")

# Ajustamos automáticamente los márgenes para que ningún texto o etiqueta quede cortado o superpuesto
figura.tight_layout()

# Finalmente, renderizamos y mostramos el gráfico en pantalla
plt.show()

# 22. Extraer los MFCC de todas las grabaciones

In [ ]:
from time import perf_counter  # Importa el cronómetro de alta precisión para medir tiempos

mfcc_por_archivo = {}  # Diccionario: nombre de archivo -> matriz MFCC completa (tramas x coeficientes)
registros_caracteristicas = []  # Lista de diccionarios; cada uno será una fila del resumen por grabación

inicio_extraccion = perf_counter()  # Marca el instante en que inicia la extracción

for contador, registro in enumerate(  # Recorre cada grabación numerándola desde 1
    datos_particionados.itertuples(index=False),  # Itera las filas del DataFrame como tuplas, sin el índice
    start=1):  # El contador empieza en 1

    matriz = extraer_mfcc(registro.ruta)  # Lee el audio de esa ruta y calcula su matriz MFCC (una fila por trama)

    mfcc_por_archivo[registro.archivo] = matriz  # Guarda la matriz completa para usarla después (por ejemplo, con los GMM)

    medias = matriz.mean(axis=0)  # Media de cada coeficiente MFCC a lo largo de todas las tramas
    desviaciones = matriz.std(axis=0, ddof=0)  # Desviación estándar poblacional (ddof=0) de cada coeficiente

    resumen = {  # Diccionario que será la fila de esta grabación
        "archivo": registro.archivo,  # Nombre del archivo de audio
        "locutor": registro.locutor,  # Locutor al que pertenece la grabación
        "particion": registro.particion,  # Partición a la que pertenece (entrenamiento, validación o prueba)
        "numero_tramas": matriz.shape[0]}  # Cantidad de tramas (número de filas de la matriz MFCC)

    for indice in range(numero_mfcc):  # Recorre cada coeficiente MFCC (0, 1, 2, ...)
        resumen[f"media_mfcc_{indice}"] = float(medias[indice])  # Agrega la media del coeficiente como columna
        resumen[f"desviacion_mfcc_{indice}"] = float(desviaciones[indice])  # Agrega la desviación del coeficiente como columna

    registros_caracteristicas.append(resumen)  # Añade la fila completa a la lista (una vez por grabación)

    if contador % 500 == 0:  # Cada 500 grabaciones procesadas...
        print("Grabaciones procesadas: ", contador)  # ...imprime el avance

# ---- Fuera del ciclo: ya se procesaron todas las grabaciones ----

caracteristicas_audio = pd.DataFrame(registros_caracteristicas)  # Convierte la lista de resúmenes en una tabla

columnas_caracteristicas = []  # Lista con los nombres de las columnas que serán las características numéricas

for indice in range(numero_mfcc):  # Recorre cada coeficiente MFCC
    columnas_caracteristicas.append(f"media_mfcc_{indice}")  # Agrega el nombre de la columna de la media

for indice in range(numero_mfcc):  # Recorre otra vez cada coeficiente MFCC
    columnas_caracteristicas.append(f"desviacion_mfcc_{indice}")  # Agrega el nombre de la columna de la desviación

if len(mfcc_por_archivo) != len(datos_particionados):  # Comprueba que haya una matriz por cada grabación del inventario
    raise ValueError(  # Si no coinciden, detiene el programa con un error
        "El número de matrices no coincide con el inventario."  # Mensaje que explica el problema
    )  # Cierra la llamada a ValueError

valores_caracteristicas = caracteristicas_audio[  # Selecciona de la tabla solo las columnas de características
    columnas_caracteristicas].to_numpy()  # Las convierte en un arreglo de NumPy (grabaciones x características)

if not np.isfinite(valores_caracteristicas).all():  # Verifica que no haya NaN ni valores infinitos
    raise ValueError(  # Si los hay, detiene el programa con un error
        "Las características contienen valores no finitos.")  # Mensaje que explica el problema

tiempo_extraccion = perf_counter() - inicio_extraccion  # Segundos totales que tomó todo el proceso

print("\nMatrices MFCC", len(mfcc_por_archivo))  # Muestra cuántas matrices MFCC se guardaron
print("Grabaciones resumidas: ", len(caracteristicas_audio))  # Muestra cuántas filas tiene la tabla de resumen
print("Características por grabación: ", len(columnas_caracteristicas))  # Muestra cuántas características tiene cada grabación (medias + desviaciones)
print(f"Tiempo de extracción: {tiempo_extraccion:.2f}segundos")  # Muestra el tiempo total con 2 decimales

# 23. Preparar y escalar las características para k-NN

In [ ]:
from sklearn.preprocessing import StandardScaler  # Importa el escalador que estandariza (media 0, desviación 1)

caracteristicas_entrenamiento = caracteristicas_audio.loc[  # Filtra la tabla completa con una condición sobre las filas
    caracteristicas_audio["particion"] == "entrenamiento"].copy()  # Se queda con las grabaciones de entrenamiento y hace una copia independiente

caracteristicas_validacion = caracteristicas_audio.loc[  # Filtra la tabla completa con una condición sobre las filas
    caracteristicas_audio["particion"] == "validacion"].copy()  # Se queda con las grabaciones de validación y hace una copia independiente

caracteristicas_prueba = caracteristicas_audio.loc[  # Filtra la tabla completa con una condición sobre las filas
    caracteristicas_audio["particion"] == "prueba"].copy()  # Se queda con las grabaciones de prueba y hace una copia independiente

x_entrenamiento = caracteristicas_entrenamiento[  # Selecciona de la partición de entrenamiento...
    columnas_caracteristicas].to_numpy()  # ...solo las columnas de características y las convierte en arreglo de NumPy

x_validacion = caracteristicas_validacion[  # Selecciona de la partición de validación...
    columnas_caracteristicas].to_numpy()  # ...solo las columnas de características y las convierte en arreglo de NumPy

x_prueba = caracteristicas_prueba[  # Selecciona de la partición de prueba...
    columnas_caracteristicas].to_numpy()  # ...solo las columnas de características y las convierte en arreglo de NumPy

y_entrenamiento = caracteristicas_entrenamiento["locutor"].to_numpy()  # Etiquetas (locutor) de entrenamiento como arreglo

y_validacion = caracteristicas_validacion["locutor"].to_numpy()  # Etiquetas (locutor) de validación como arreglo

y_prueba = caracteristicas_prueba["locutor"].to_numpy()  # Etiquetas (locutor) de prueba como arreglo

escalador_knn = StandardScaler()  # Crea el escalador, todavía sin ajustar

x_entrenamiento_escalado = escalador_knn.fit_transform(x_entrenamiento)  # Calcula media y desviación SOLO con entrenamiento y lo escala

x_validacion_escalado = escalador_knn.transform(x_validacion)  # Escala validación con la media y desviación de entrenamiento

x_prueba_escalado = escalador_knn.transform(x_prueba)  # Escala prueba con la media y desviación de entrenamiento

for matri in [  # Recorre las tres matrices escaladas
    x_entrenamiento_escalado,  # Matriz de entrenamiento escalada
    x_validacion_escalado,  # Matriz de validación escalada
    x_prueba_escalado]:  # Matriz de prueba escalada

    if not np.isfinite(matri).all():  # Verifica que no haya NaN ni valores infinitos
        raise ValueError("Las características escaladas contienen valores no finitos.")  # Si los hay, detiene el programa con un error

print("Entrenamiento: ", x_entrenamiento_escalado.shape)  # Muestra (grabaciones, características) de entrenamiento
print("Validación: ", x_validacion_escalado.shape)  # Muestra (grabaciones, características) de validación
print("Prueba: ", x_prueba_escalado.shape)  # Muestra (grabaciones, características) de prueba

print("Locutores en entrenamiento: ", len(np.unique(y_entrenamiento)))  # Cuenta cuántos locutores distintos hay en entrenamiento

print("Grabaciones usadas para ajustar el escalador: ", escalador_knn.n_samples_seen_)  # Muestra cuántas muestras vio el escalador al ajustarse


# 24. Comparar configuraciones de k-NN en validación

In [ ]:
from sklearn.neighbors import KNeighborsClassifier  # Importa el clasificador de k vecinos más cercanos
from sklearn.metrics import accuracy_score, f1_score  # Importa las métricas de exactitud y F1

opciones_vecinos = [1, 3, 5, 7, 9, 21]  # Valores de k (número de vecinos) que se van a probar
opciones_pesos = ["uniform", "distance"]  # "uniform": todos los vecinos pesan igual; "distance": los más cercanos pesan más
opciones_p = [1, 2]  # Parámetro p de Minkowski: 1 = distancia Manhattan, 2 = distancia Euclidiana

resultados_knn = []  # Lista donde se guardan las métricas de cada combinación probada

mejor_modelo_knn = None  # Aquí se guardará el mejor modelo encontrado (aún no hay ninguno)
mejor_puntuacion_knn = (-1.0, -1.0)  # Mejor (F1 macro, exactitud) hasta ahora; empieza en -1 para que cualquier resultado lo supere
mejores_parametros_knn = None  # Aquí se guardarán los hiperparámetros del mejor modelo

for vecinos in opciones_vecinos:  # Ciclo externo: prueba cada valor de k
    for pesos in opciones_pesos:  # Ciclo intermedio: prueba cada tipo de ponderación
        for valor_p in opciones_p:  # Ciclo interno: prueba cada valor de p

            modelo = KNeighborsClassifier(  # Crea un clasificador KNN con la combinación actual
                n_neighbors=vecinos,  # Número de vecinos a considerar
                weights=pesos,  # Tipo de ponderación de los vecinos
                metric="minkowski",  # Métrica de distancia (generaliza Manhattan y Euclidiana)
                p=valor_p,  # Potencia de la distancia de Minkowski
                algorithm="brute",  # Búsqueda por fuerza bruta: compara contra todas las muestras
                n_jobs=-1)  # Usa todos los núcleos del procesador

            modelo.fit(x_entrenamiento_escalado, y_entrenamiento)  # "Entrena" el modelo (KNN solo memoriza los datos de entrenamiento)

            predicciones = modelo.predict(x_validacion_escalado)  # Predice el locutor de cada grabación de validación

            exactitud = accuracy_score(y_validacion, predicciones)  # Proporción de predicciones correctas

            f1_macro = f1_score(  # Calcula el F1-score
                y_validacion,  # Etiquetas reales
                predicciones,  # Etiquetas predichas
                average="macro",  # Promedia el F1 de cada locutor dándoles el mismo peso
                zero_division=0)  # Si hay división entre cero, asigna 0 en lugar de advertencia

            resultados_knn.append({  # Guarda el resultado de esta combinación
                "vecinos": vecinos,  # Valor de k usado
                "pesos": pesos,  # Ponderación usada
                "p": valor_p,  # Valor de p usado
                "exactitud_validacion": exactitud,  # Exactitud obtenida en validación
                "f1_macro_validacion": f1_macro})  # F1 macro obtenido en validación

            puntuacion = (f1_macro, exactitud)  # Tupla de puntuación: el F1 macro manda y la exactitud desempata

            if puntuacion > mejor_puntuacion_knn:  # Compara las tuplas elemento por elemento (primero F1, luego exactitud)
                mejor_puntuacion_knn = puntuacion  # Actualiza la mejor puntuación
                mejor_modelo_knn = modelo  # Guarda este modelo como el mejor

                mejores_parametros_knn = {  # Guarda los hiperparámetros del mejor modelo
                    "n_neighbors": vecinos,  # Mejor valor de k
                    "weights": pesos,  # Mejor ponderación
                    "p": valor_p}  # Mejor valor de p

# ---- Fuera de los ciclos: ya se probaron todas las combinaciones ----

tabla_resultados_knn = pd.DataFrame(resultados_knn)  # Convierte la lista de resultados en una tabla

tabla_resultados_knn = tabla_resultados_knn.sort_values(  # Ordena la tabla de resultados
    by=["f1_macro_validacion", "exactitud_validacion"],  # Primero por F1 macro y, en empate, por exactitud
    ascending=False,  # De mayor a menor (la mejor combinación queda arriba)
    kind="stable").reset_index(drop=True)  # Orden estable; reinicia el índice descartando el anterior

print("Configuraciones evaluadas:", len(tabla_resultados_knn))  # Muestra cuántas combinaciones se probaron

print(tabla_resultados_knn.to_string(  # Imprime la tabla completa como texto
    index=False,  # Sin mostrar el índice
    float_format="{:.4f}".format))  # Los decimales se muestran con 4 dígitos

print("\nMejores parámetros: ", mejores_parametros_knn)  # Muestra los hiperparámetros ganadores
print(f"F1 macro en validación: {mejor_puntuacion_knn[0]:.4f}")  # Muestra el F1 macro del mejor modelo con 4 decimales
print(f"Exactitud en validación: {mejor_puntuacion_knn[1]:.4f}")  # Muestra la exactitud del mejor modelo con 4 decimales


# 25. Identificar los errores de k-NN en validación

In [ ]:
predicciones_validacion_knn = mejor_modelo_knn.predict(  # Usa el mejor modelo KNN para predecir...
    x_validacion_escalado)  # ...el locutor de cada grabación de validación

revision_validacion_knn = caracteristicas_validacion[  # Toma la tabla de características de validación...
    ["archivo", "locutor"]].copy()  # ...y se queda solo con el archivo y el locutor real, en una copia independiente

revision_validacion_knn["locutor_predicho"] = (predicciones_validacion_knn)  # Agrega la columna con el locutor predicho por el modelo

revision_validacion_knn["acierto"] = (  # Agrega una columna booleana de acierto
    revision_validacion_knn["locutor"] == revision_validacion_knn["locutor_predicho"])  # True si el locutor real coincide con el predicho

revision_validacion_knn = revision_validacion_knn.merge(  # Une esta tabla con otra por una columna en común
    datos_validacion[["archivo", "duracion_segundos"]],  # Tabla con el nombre del archivo y la duración de cada audio
    on="archivo",  # La unión se hace por el nombre del archivo
    how="left",  # Conserva todas las filas de la tabla de revisión
    validate="one_to_one")  # Verifica que cada archivo aparezca una sola vez en ambas tablas

revision_validacion_knn["capitulo"] = (  # Agrega la columna del capítulo
    revision_validacion_knn["archivo"].str.split("-").str[1])  # Separa el nombre del archivo por guiones y toma el segundo elemento

errores_validacion_knn = revision_validacion_knn.loc[  # Filtra la tabla de revisión
    ~revision_validacion_knn["acierto"]].copy()  # Se queda solo con las filas donde NO hubo acierto (errores), en una copia

cantidad_aciertos = int(revision_validacion_knn["acierto"].sum())  # Suma los True (cada uno vale 1) para contar los aciertos

print("Grabaciones evaluadas:", len(revision_validacion_knn))  # Muestra el total de grabaciones evaluadas
print("Cantidad de aciertos: ", cantidad_aciertos)  # Muestra cuántas se clasificaron bien
print("Cantidad de errores: ", len(errores_validacion_knn))  # Muestra cuántas se clasificaron mal

columnas_revision_knn = [  # Lista de columnas que se mostrarán en la tabla de errores
    "archivo",  # Nombre del archivo de audio
    "locutor",  # Locutor real
    "locutor_predicho",  # Locutor que predijo el modelo
    "capitulo",  # Capítulo al que pertenece la grabación
    "duracion_segundos"]  # Duración del audio en segundos

if not errores_validacion_knn.empty:  # Si hubo al menos un error...
    print(errores_validacion_knn[columnas_revision_knn].to_string(  # ...imprime la tabla de errores solo con esas columnas
        index=False,  # Sin mostrar el índice
        float_format="{:.2f}".format))  # Los decimales se muestran con 2 dígitos
else:  # Si no hubo ningún error...
    print("No hubo errores en validación.")  # ...muestra este mensaje



# 26. Agrupar las tramas de entrenamiento para GMM

In [ ]:
tramas_entrenamiento_por_locutor = {}  # Diccionario: locutor -> matriz con todas sus tramas MFCC de entrenamiento
registros_tramas_gmm = []  # Lista de diccionarios; cada uno será una fila del resumen por locutor

for locutor, grupo in datos_entrenamiento.groupby("locutor", sort=True):  # Agrupa las grabaciones por locutor (ordenado) y recorre cada grupo

    matrices_locutor = []  # Lista temporal con las matrices MFCC de las grabaciones de este locutor

    for archivo in grupo["archivo"]:  # Recorre los archivos de audio de este locutor
        matrices_locutor.append(  # Agrega a la lista...
            mfcc_por_archivo[archivo])  # ...la matriz MFCC de ese archivo, buscándola por su nombre

    tramas_locutor = np.concatenate(matrices_locutor, axis=0)  # Apila todas las matrices una debajo de otra (une las tramas de todas sus grabaciones)

    if tramas_locutor.ndim != 2:  # Verifica que el resultado sea una matriz de 2 dimensiones (tramas x coeficientes)
        raise ValueError("Se esperaba una matriz bidimensional.")  # Si no, detiene el programa con un error

    if tramas_locutor.shape[1] != numero_mfcc:  # Verifica que el número de columnas sea igual al número de coeficientes MFCC
        raise ValueError("El número de MFCC no coincide.")  # Si no, detiene el programa con un error

    if not np.isfinite(tramas_locutor).all():  # Verifica que no haya NaN ni valores infinitos
        raise ValueError("Las tramas contienen valores no finitos.")  # Si los hay, detiene el programa con un error

    tramas_entrenamiento_por_locutor[locutor] = (tramas_locutor)  # Guarda las tramas del locutor en el diccionario

    registros_tramas_gmm.append({  # Agrega una fila de resumen para este locutor
        "locutor": locutor,  # Identificador del locutor
        "grabaciones_entrenamiento": len(grupo),  # Cantidad de grabaciones de entrenamiento que tiene
        "tramas_entrenamiento": tramas_locutor.shape[0]})  # Cantidad total de tramas (número de filas de la matriz)

# ---- Fuera del ciclo: ya se procesaron todos los locutores ----

resumen_tramas_gmm = pd.DataFrame(registros_tramas_gmm)  # Convierte la lista de resúmenes en una tabla

total_tramas_esperadas = sum(mfcc_por_archivo[archivo].shape[0]  # Suma el número de tramas de cada matriz MFCC...
                             for archivo in datos_entrenamiento["archivo"])  # ...recorriendo todos los archivos de entrenamiento

total_tramas_agrupadas = int(resumen_tramas_gmm["tramas_entrenamiento"].sum())  # Suma las tramas ya agrupadas por locutor, como entero

if total_tramas_agrupadas != total_tramas_esperadas:  # Comprueba que no se haya perdido ni duplicado ninguna trama
    raise ValueError("El total de tramas no coincide.")  # Si no coinciden, detiene el programa con un error

print("Locutores esperados:",  # Muestra cuántos locutores...
      len(tramas_entrenamiento_por_locutor))  # ...hay en el diccionario
print("Total de tramas de entrenamiento: ", total_tramas_agrupadas)  # Muestra el total de tramas de entrenamiento
print("Mínimo de tramas por locutor:",  # Muestra el menor número de tramas que tiene un locutor...
      resumen_tramas_gmm["tramas_entrenamiento"].min())  # ...calculado sobre la tabla de resumen
print("Máximo de tramas por locutor:",  # Muestra el mayor número de tramas que tiene un locutor...
      resumen_tramas_gmm["tramas_entrenamiento"].max())  # ...calculado sobre la tabla de resumen
print("MFCC por trama:", numero_mfcc)  # Muestra cuántos coeficientes MFCC tiene cada trama





# 27. Definir las configuraciones de GMM

In [ ]:
from sklearn.mixture import GaussianMixture  # Importa el modelo de mezclas gaussianas (GMM)

opciones_componentes_gmm = [8, 16, 32]  # Números de componentes gaussianas que se van a comparar

parametros_base_gmm = {  # Diccionario con los hiperparámetros comunes a todos los modelos
    "covariance_type": "diag",  # Covarianza diagonal: más rápida y con menos parámetros que la completa
    "tol": 2e-3,  # Tolerancia: el EM se detiene cuando la mejora es menor a este valor
    "reg_covar": 1e-6,  # Valor pequeño que se suma a la covarianza para evitar inestabilidad numérica
    "max_iter": 200,  # Máximo de iteraciones del algoritmo EM
    "n_init": 2,  # Número de inicializaciones; se conserva la mejor
    "init_params": "kmeans",  # Los parámetros iniciales se calculan con K-means
    "random_state": semilla}  # Semilla aleatoria para que los resultados sean reproducibles

minimo_tramas_gmm = min(matriz.shape[0]  # Obtiene el menor número de tramas...
                        for matriz in tramas_entrenamiento_por_locutor.values())  # ...entre las matrices de todos los locutores

if max(opciones_componentes_gmm) > minimo_tramas_gmm:  # Comprueba si el mayor número de componentes supera las tramas del locutor con menos datos
    raise ValueError("Hay más componentes que tramas disponibles.")  # Si es así, detiene el programa con un error

print("Componentes por comparar:", opciones_componentes_gmm)  # Muestra las cantidades de componentes a probar
print("Tipo de covarianza:", parametros_base_gmm["covariance_type"])  # Muestra el tipo de covarianza elegido
print("Máximo de iteraciones: ", parametros_base_gmm["max_iter"])  # Muestra el máximo de iteraciones del EM
print("Inicializacion por modelo:", parametros_base_gmm["n_init"])  # Muestra cuántas inicializaciones usa cada modelo
print("Semilla:", parametros_base_gmm["random_state"])  # Muestra la semilla aleatoria
print("Total de modelos por entrenar:",  # Muestra cuántos modelos se entrenarán en total...
      len(tramas_entrenamiento_por_locutor) * len(opciones_componentes_gmm))  # ...(número de locutores x número de configuraciones)

# 28. Entrenar y evaluar los GMM en validación

In [ ]:
modelos_gmm_por_configuracion = {}  # Diccionario: guarda, por cada número de componentes, los modelos de todos los locutores
resultados_gmm = []  # Lista donde se guardan las métricas de validación de cada configuración
registros_entrenamiento_gmm = []  # Lista donde se guarda el detalle del entrenamiento de cada modelo individual

locutores_gmm = sorted(tramas_entrenamiento_por_locutor.keys())  # Lista ordenada con los identificadores de los locutores
etiquetas_locutores_gmm = np.array(locutores_gmm)  # Mismo listado como arreglo de NumPy para indexarlo con argmax
y_validacion_gmm = datos_validacion["locutor"].to_numpy()  # Etiquetas reales (locutor) de las grabaciones de validación

for componentes in opciones_componentes_gmm:  # Prueba cada cantidad de componentes gaussianas candidata
    print(f"\nConfiguración: {componentes} componentes", flush=True)  # Informa qué configuración se está procesando

    modelos_actuales = {}  # Diccionario temporal: locutor -> modelo GMM de esta configuración
    inicio_entrenamiento = perf_counter()  # Marca el instante en que inicia el entrenamiento

    for contador, locutor in enumerate(locutores_gmm, start=1):  # Recorre los locutores numerándolos desde 1
        modelo = GaussianMixture(  # Crea un GMM vacío para este locutor
            n_components=componentes,  # Número de gaussianas de la mezcla
            **parametros_base_gmm)  # Desempaqueta el resto de hiperparámetros comunes (tipo de covarianza, iteraciones, etc.)
        modelo.fit(tramas_entrenamiento_por_locutor[locutor])  # Entrena el GMM con las tramas MFCC de ese locutor

        modelos_actuales[locutor] = modelo  # Guarda el modelo entrenado asociado a su locutor
        registros_entrenamiento_gmm.append({  # Agrega un registro con los datos del entrenamiento
            "componentes": componentes,  # Configuración usada
            "locutor": locutor,  # Locutor al que pertenece el modelo
            "convergio": modelo.converged_,  # True si el algoritmo EM convergió
            "iteraciones": modelo.n_iter_})  # Número de iteraciones que necesitó el EM

        if contador % 5 == 0:  # Cada 5 modelos entrenados...
            print(  # ...imprime el avance
                f"Modelos entrenados: {contador}/{len(locutores_gmm)}",  # Muestra "entrenados/total"
                flush=True)  # Fuerza la salida inmediata en pantalla

    tiempo_entrenamiento = perf_counter() - inicio_entrenamiento  # Segundos totales que tomó entrenar esta configuración

    modelos_gmm_por_configuracion[componentes] = modelos_actuales  # Guarda todos los modelos de esta configuración

    inicio_validacion = perf_counter()  # Marca el instante en que inicia la validación
    predicciones_actuales = []  # Lista donde se guardan los locutores predichos

    for contador, registro in enumerate(  # Recorre cada grabación de validación numerándola desde 1
            datos_validacion.itertuples(index=False),  # Itera las filas del DataFrame como tuplas, sin el índice
            start=1):  # El contador empieza en 1
        matriz = mfcc_por_archivo[registro.archivo]  # Obtiene la matriz de MFCC del archivo de audio actual

        puntuaciones = np.array([  # Arreglo con una puntuación por cada locutor
            modelos_actuales[locutor].score(matriz)  # Log-verosimilitud promedio de la grabación bajo el modelo de ese locutor
            for locutor in locutores_gmm])  # Se calcula para todos los locutores, en el mismo orden

        if not np.isfinite(puntuaciones).all():  # Verifica que ninguna puntuación sea NaN o infinita
            raise ValueError(  # Si hay alguna, detiene el programa con un error
                f"Puntuaciones no finitas en {registro.archivo}.")  # Indica en qué archivo ocurrió el problema

        indice_mejor = int(np.argmax(puntuaciones))  # Posición del locutor con la puntuación más alta
        predicciones_actuales.append(  # Guarda la predicción de esta grabación
            etiquetas_locutores_gmm[indice_mejor])  # Convierte la posición en el identificador del locutor

        if contador % 100 == 0:  # Cada 100 grabaciones evaluadas...
            print(  # ...imprime el avance
                f"Grabaciones de validación evaluadas: "  # Primera parte del mensaje
                f"{contador}/{len(datos_validacion)}",  # Muestra "evaluadas/total"
                flush=True)  # Fuerza la salida inmediata en pantalla

    tiempo_validacion = perf_counter() - inicio_validacion  # Segundos totales que tomó la validación

    exactitud = accuracy_score(  # Calcula la proporción de predicciones correctas
        y_validacion_gmm,  # Etiquetas reales
        predicciones_actuales)  # Etiquetas predichas por los GMM

    f1_macro = f1_score(  # Calcula el F1-score
        y_validacion_gmm,  # Etiquetas reales
        predicciones_actuales,  # Etiquetas predichas
        average="macro",  # Promedia el F1 de cada locutor dándoles el mismo peso
        zero_division=0)  # Si hay división entre cero, asigna 0 en lugar de advertencia

    modelos_sin_convergencia = sum(  # Cuenta cuántos modelos no convergieron
        not modelo.converged_  # True (1) cuando el modelo no convergió
        for modelo in modelos_actuales.values())  # Revisa todos los modelos de la configuración actual

    resultados_gmm.append({  # Agrega el resumen de esta configuración
        "componentes": componentes,  # Número de componentes probado
        "exactitud_validacion": exactitud,  # Exactitud obtenida en validación
        "f1_macro_validacion": f1_macro,  # F1 macro obtenido en validación
        "modelos_sin_convergencia": modelos_sin_convergencia,  # Cantidad de modelos que no convergieron
        "tiempo_entrenamiento_s": tiempo_entrenamiento,  # Tiempo de entrenamiento en segundos
        "tiempo_validacion_s": tiempo_validacion})  # Tiempo de validación en segundos

    print(f"F1 macro: {f1_macro:.4f} | "  # Muestra el F1 macro con 4 decimales
          f"Exactitud: {exactitud:.4f} | "  # Muestra la exactitud con 4 decimales
          f"Modelos sin convergencia: {modelos_sin_convergencia}",  # Muestra cuántos modelos no convergieron
          flush=True)  # Fuerza la salida inmediata en pantalla

# ---- Fuera del ciclo: ya se probaron todas las configuraciones ----

tabla_entrenamiento_gmm = pd.DataFrame(  # Convierte los registros de entrenamiento en una tabla
    registros_entrenamiento_gmm)  # Lista de diccionarios con el detalle de cada modelo

tabla_resultados_gmm = pd.DataFrame(resultados_gmm)  # Convierte los resultados por configuración en una tabla

tabla_resultados_gmm = tabla_resultados_gmm.sort_values(  # Ordena la tabla de resultados
    by=["f1_macro_validacion", "exactitud_validacion"],  # Primero por F1 macro y, en empate, por exactitud
    ascending=False,  # De mayor a menor (la mejor configuración queda arriba)
    kind="stable").reset_index(drop=True)  # Orden estable; reinicia el índice descartando el anterior

mejores_componentes_gmm = int(  # Número de componentes de la mejor configuración, como entero
    tabla_resultados_gmm.iloc[0]["componentes"])  # Toma la primera fila (la mejor) y su columna "componentes"

mejores_modelos_gmm = modelos_gmm_por_configuracion[  # Recupera los modelos de la mejor configuración
    mejores_componentes_gmm]  # Se busca por su número de componentes

print("\nResultados de GMM:")  # Título de la tabla de resultados
print(tabla_resultados_gmm.to_string(  # Imprime la tabla completa como texto
    index=False,  # Sin mostrar el índice
    float_format="{:.4f}".format))  # Los decimales se muestran con 4 dígitos

print("\nComponentes seleccionados:", mejores_componentes_gmm)  # Muestra cuántos componentes ganaron

# Representación digital de voz
* Frecuencia de muestreo: cantidad de muestras que se toman por unidad de tiempo para convertir la señal analógica en señal digital.
* Profundidad de bits: cantidad de bits que pueden mostrar color o amplitud en imágenes o sonidos. Indica el nivel de detalle y precisión al grabar o mostrar contenido digital.
* Canales: representación de una señal de audio en los procesos de grabación, reproducción o mezcla
* Cuantificación: proceso en el cual se asignan valores finitos, de un conjunto de niveles definidos previamente, a las muestras tomadas de la señal analógica.
* Teorema de Nyquist-Shannon: establece que para reconstruir una señal analógica continua en el tiempo sin perder información, la tasa de muestreo debe ser mayor al doble de su frecuencia máxima.

# Corpus de voz
LibriSpeech
* Características: conjunto de datos con grabaciones de audio libros de dominio público.
* Estructura: 1, 000 horas de grabaciones de voz en inglés de forma leída. Los audios estan muestreados a 16KHz y cuentan con transcripciones de texto alineadas.
* Aplicaciones: entrenamiento de modelos ASR, clasificación e identificación de hablantes, análisis de audio y emociones.
* Limitaciones: dominio exclusivo del habla leída, falta de ruido ambiental real, sesgo lingüístico y de acentos, ausencia de dialogo superpuesto y las transcirpciones están en texto plano, sin acentos y en minúsculas.

Common voice
* Características: base de datos multilingüe y de dominio público, los datos se obtienen de la participación voluntaria de las personas, reúne muestras de voz en cientos de idiomas. Mitigación de sesgos en la IA, sirve para entrenar modelos de aprendizaje automático.
* Estructura: frases cortas preestablecidas, respuestas libres e improvisadas a preguntas casuales, reflejan conversaciones desordenadas de la vida diaria.
* Aplicaciones: sistemas de reconocimiento automático del habla, modelo de lenguaje y transfer learning, sistemas de texto a voz
* Limitaciones: calidad del audio, distribución demográfica sesgada, metadatos incompletos, desequilibrio entre idiomas.

VoxCeleb
* Características:conjunto de datos a gran escala de audio y video diseñado para el reconocimiento de locutores y la biometría de voz en condiciones reales. Extraído de videos de YouTube, incluye ruido del mundo real, abarca múltiples etnias, acentos, edades y profesionales.
* Estructura: esta estructurado de forma jerárquica para facilitar el procesamiento tanto de audio como de video: nivel 1 (identidad), nivel 2 (video de origen), nivel 3 (fragmentos). Más de 1 millón de enunciados, más de 2000 horas de audio.
* Aplicaciones: verificación del locutor, identificación del locutor, conversión de texto a voz y clonación de voz, modelos multimodales.
* Limitaciones: sesgo demográfico y geográfico, sesgo de edad, calidad variable del audio, falsos positivos en la recolección.

# Librerías de procesamiento de audio

Librosa
* Funciones principales: carga de audio, representaciones espectrales y manipulación de señales.
* Características: es una librería para el procesamiento de audio y música. Proporciona los algoritmos y herramientas fundamentales necesarios para construir sistemas de recuperación de información musical.

SoundFile
* Funciones principales: funciones de lectura/escritura, procesamiento de bloques, los archivos de audio se pueden abrir como objetos, se pueden leer y escribir fragmentos de audio sin tocar el sistema de archivos.
* Características: puede leer y escribir archivos de audio

* Diferencias entre ambos: SoundFile se utiliza para leer y escribir archivos de audio, mientras que Librosa es una librería para el análisis musical y extracción de características de audio.

# Escala de MEL y MFCC

# ¿Qué es la escala de Mel?

La escala de Mel es una representación perceptual de la frecuencia que busca aproximarse a la manera en que el oído humano distingue las alturas de los sonidos. A diferencia de una escala lineal expresada en hercios (Hz), donde los intervalos corresponden a diferencias constantes de frecuencia, la escala de Mel representa las frecuencias de acuerdo con una aproximación de la percepción auditiva.

En términos generales, el sistema auditivo distingue con mayor resolución ciertos intervalos de frecuencias bajas, mientras que las diferencias entre frecuencias altas se representan de forma más comprimida. Por ello, en el procesamiento de voz se emplea una distribución no lineal de las bandas de frecuencia.

---

## Fórmula de conversión de hertz a Mel

Una de las expresiones utilizadas para convertir una frecuencia lineal $f$, medida en hertz, a la escala de Mel es:

$$
m = 2595 \log_{10}\left(1+\frac{f}{700}\right)
$$

Donde:
- $m$: frecuencia expresada en Mels.
- $f$: frecuencia original en hertz (Hz).
- $\log_{10}$: logaritmo en base 10.
- $700$: constante de la aproximación empleada.

---

## Banco de filtros de Mel

Un banco de filtros de Mel es un conjunto de filtros pasabanda que divide el espectro de una señal de audio en diferentes regiones de frecuencia. Sus frecuencias centrales se distribuyen uniformemente en la escala de Mel, no en la escala convencional de hertz.

Los filtros triangulares son una de las configuraciones más utilizadas para calcular los MFCC. Cada filtro pondera los componentes espectrales que se encuentran dentro de su región y produce un valor que representa la energía o magnitud espectral de dicha banda.

### 1. Distribución de las frecuencias de los filtros

Si se utilizan $M$ filtros, las frecuencias se distribuyen en el dominio Mel mediante:

$$
m_i = m_{\min} + i \frac{m_{\max}-m_{\min}}{M+1}
$$

Donde:
- $m_i$ es la frecuencia del punto $i$ en la escala de Mel.
- $m_{\min}$ y $m_{\max}$ son los límites de frecuencia transformados a Mel.
- $M$ es el número de filtros.
- $i$ identifica cada punto de la distribución.

Para obtener las frecuencias correspondientes en hertz, se aplica la transformación inversa:

$$
f_i = 700\left(e^{m_i/1127}-1\right)
$$

De esta manera, los filtros quedan distribuidos uniformemente en Mel, pero sus anchos de banda en hertz aumentan a medida que se avanza hacia frecuencias superiores.

### 2. Función matemática de un filtro triangular

La respuesta de frecuencia del filtro $m$ puede definirse como:

$$
H_m(k)=
\begin{cases}
0, & k < f_{m-1}\\[4pt]
\dfrac{k-f_{m-1}}{f_m-f_{m-1}}, & f_{m-1} \leq k \leq f_m\\[8pt]
\dfrac{f_{m+1}-k}{f_{m+1}-f_m}, & f_m < k \leq f_{m+1}\\[8pt]
0, & k > f_{m+1}
\end{cases}
$$

Donde:
- $H_m(k)$ representa la respuesta del filtro $m$ en el índice de frecuencia $k$.
- $f_{m-1}$ es la frecuencia inicial del filtro.
- $f_m$ es su frecuencia central.
- $f_{m+1}$ es su frecuencia final.

Esta función tiene tres regiones principales: una zona de crecimiento lineal, un máximo en la frecuencia central y una zona de decrecimiento lineal. La forma triangular permite que las frecuencias próximas al centro tengan mayor peso y que las frecuencias cercanas a los extremos contribuyan en menor medida.

### 3. Cálculo de la energía de cada filtro

Antes de aplicar el banco de filtros, normalmente se transforma cada segmento de audio al dominio de la frecuencia. Para una señal discreta $x[n]$, la transformada discreta de Fourier se expresa como:

$$
X[k] = \sum_{n=0}^{N-1} x[n]e^{-j2\pi kn/N}
$$

Donde $N$ es el número de muestras del segmento y $k$ es el índice de frecuencia.

En el análisis por ventanas, la señal se multiplica por una función de ventana $w[n]$, por ejemplo, la ventana de Hamming:

$$
w[n] = 0.54 - 0.46\cos\left(\frac{2\pi n}{N-1}\right)
$$

para $0 \leq n < N$.

La señal utilizada para calcular el espectro queda definida como:

$$
x_w[n] = x[n]w[n]
$$

Esta operación reduce las discontinuidades en los extremos del segmento. Una vez obtenido el espectro, la energía de la banda $m$ puede calcularse mediante:

$$
E_m = \sum_{k=0}^{N-1} |X[k]|^2 H_m[k]
$$

Aquí, $|X[k]|^2$ representa el espectro de potencia y $H_m[k]$ es la respuesta del filtro.

El resultado es un vector de $M$ valores:

$$
\mathbf{E} = [E_0, E_1, \ldots, E_{M-1}]
$$

Cada valor resume la contribución espectral de una banda de Mel.

---

## Transformación logarítmica

### 1. Fundamentos

Después de calcular las salidas del banco de filtros, se aplica una transformación logarítmica. Esta etapa es importante porque permite comprimir el rango dinámico de las energías y representa de forma aproximada la relación no lineal entre la intensidad física del sonido y la percepción auditiva.

### 2. Fórmula de la transformación logarítmica

La salida logarítmica del filtro $m$ se define como:

$$
L[m] = \ln(E_m+\varepsilon)
$$

Donde:
- $L[m]$ es la salida logarítmica del filtro $m$.
- $E_m$ es la energía calculada por el filtro.
- $\ln$ es el logaritmo natural.
- $\varepsilon > 0$ es una constante pequeña que evita calcular el logaritmo de cero.

El resultado es un vector de valores logarítmicos:

$$
\mathbf{L} = [L[0], L[1], \ldots, L[M-1]]
$$

### 3. ¿Por qué se utiliza el logaritmo?

Supongamos que dos bandas tienen energías $E_1=10$ y $E_2=1000$. Sus magnitudes son muy diferentes, pero el logaritmo natural las convierte en valores aproximadamente iguales a $2.30$ y $6.91$, respectivamente. Esto comprime las diferencias entre valores elevados sin eliminar completamente las diferencias originales.

---

## Transformada Discreta del Coseno (DCT)

### 1. Fundamentos

La transformada discreta del coseno (DCT) convierte una secuencia de valores en una representación basada en funciones cosenoidales. En el cálculo de MFCC, se aplica al vector obtenido después de la transformación logarítmica para obtener una representación cepstral compacta y reducir la correlación entre las características. Esto facilita el uso de los coeficientes como entrada a algoritmos de clasificación.

### 2. Fórmula de la DCT de tipo II

Una expresión habitual de la DCT de tipo II, sin normalización explícita, es:

$$
C_n = \sum_{m=0}^{M-1} L[m]\cos\left[\frac{\pi n}{M}\left(m+\frac{1}{2}\right)\right]
$$

Donde:
- $C_n$ es el coeficiente cepstral de índice $n$.
- $L[m]$ es el valor logarítmico obtenido del filtro $m$.
- $M$ es el número de filtros de Mel.
- $n$ identifica el coeficiente que se calcula.

Para una versión ortonormal de la DCT-II, se utiliza la normalización:

$$
C_n = \alpha(n) \sum_{m=0}^{M-1} L[m]\cos\left[\frac{\pi n}{M}\left(m+\frac{1}{2}\right)\right]
$$

con:

$$
\alpha(n)=
\begin{cases}
\sqrt{\frac{1}{M}}, & n=0\\[5pt]
\sqrt{\frac{2}{M}}, & n>0
\end{cases}
$$

### 3. ¿Qué representan los coeficientes?

Los coeficientes MFCC codifican la forma general del espectro logarítmico en una representación cepstral. 
- Los **primeros coeficientes** describen variaciones suaves y amplias de la envolvente espectral.
- Los **coeficientes de orden superior** representan variaciones progresivamente más detalladas.



# Estandarización mediante StandardScaler

La estandarización mediante StandardScaler es una técnica de preprocesamiento que transforma cada característica para que tenga media cero y desviación estándar uno, calculadas a partir de los datos utilizados para ajustar el transformador. Su propósito es evitar que las diferencias de magnitud entre características influyan de forma desproporcionada en ciertos algoritmos.

Este procedimiento debe aplicarse siguiendo un orden correcto. Si se utilizan datos de prueba para calcular los parámetros de estandarización antes de evaluar el modelo, se produce una forma de contaminación entre los conjuntos de entrenamiento y prueba, conocida como *data leakage* o fuga de información. 

---

## ¿Qué es StandardScaler?

StandardScaler es un procedimiento de estandarización basado en la puntuación z o *z-score*. Para cada característica, resta su media y divide el resultado entre su desviación estándar.

La fórmula general es:

$$
z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

Donde:
- $x_{ij}$: valor original de la característica $j$ en la muestra $i$.
- $\mu_j$: media de la característica $j$.
- $\sigma_j$: desviación estándar de esa característica.
- $z_{ij}$: valor estandarizado.

La transformación centra los datos alrededor de cero y ajusta su dispersión a una escala común. No obliga a que los valores se encuentren en un intervalo fijo ni garantiza que los datos tengan una distribución normal.

---

## Cálculo de la media

La media de una característica $j$, utilizando $n$ muestras de entrenamiento, se calcula como:

$$
\mu_j = \frac{1}{n} \sum_{i=1}^{n} x_{ij}
$$

La media representa el valor promedio de esa característica. En un sistema de MFCC, por ejemplo, se calcula una media independiente para cada coeficiente, utilizando los datos de entrenamiento. 
---

## Cálculo de la desviación estándar

La desviación estándar mide cuánto se dispersan los valores respecto a su media. Para la convención utilizada por StandardScaler, se expresa como:

$$
\sigma_j = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} (x_{ij} - \mu_j)^2 }
$$

Al dividir entre la desviación estándar, se reduce el efecto de las diferencias de dispersión entre características.

$$
z = \frac{14 - 10}{2} = 2
$$

Esto significa que el valor estandarizado se encuentra dos desviaciones estándar por encima de la media de esa característica.

---

## ¿Por qué es importante escalar los MFCC?

La estandarización puede ofrecer varios beneficios, dependiendo del algoritmo utilizado:

- **Equilibrio entre características:** Cuando un algoritmo utiliza distancias o magnitudes numéricas, una característica con mayor dispersión puede influir desproporcionadamente en el cálculo. Estandarizar permite que las características contribuyan en una escala comparable.
- **Mejor comportamiento en modelos sensibles a la escala:** Los algoritmos como SVM, k vecinos más cercanos (KNN), regresión logística regularizada y determinados modelos neuronales suelen beneficiarse de una representación escalada. En una SVM con kernel de base radial, por ejemplo, las diferencias de escala pueden modificar las distancias entre muestras y, por lo tanto, la separación aprendida por el modelo.
- **Optimización del entrenamiento:** En modelos entrenados mediante métodos de gradiente, la estandarización puede mejorar el condicionamiento numérico y facilitar una convergencia más eficiente.
- **Comparación más consistente:** Al mantener el mismo procedimiento de transformación entre entrenamiento y predicción, se evita que las diferencias de escala introduzcan variaciones artificiales en el procesamiento.

Estos beneficios no significan que StandardScaler sea obligatorio en todos los casos. Los modelos basados en árboles de decisión y, generalmente, los bosques aleatorios son mucho menos sensibles a la escala de las variables. Conviene evaluar el efecto de la estandarización mediante validación experimental.

---

## Importancia de prevenir la fuga de información (data leakage)

### Definición

La fuga de información ocurre cuando durante el entrenamiento o la evaluación de un modelo se utiliza información que no debería estar disponible en esa etapa. Como resultado, el rendimiento estimado puede ser poco realista y no representar la capacidad del sistema para trabajar con datos nuevos.

En un proyecto que utiliza MFCC para clasificar sonidos, esta situación puede ocurrir si primero se estandarizan todos los vectores de características y después se realiza la separación entre entrenamiento y prueba.

### ¿Cómo se produce la fuga de información al usar StandardScaler?

Supongamos que se dispone de 1,000 grabaciones de audio y se desea construir un modelo para clasificar dos categorías de sonidos.

El conjunto de datos contiene los vectores MFCC de cada grabación. Si se calcula la media y la desviación estándar con las 1,000 grabaciones antes de separar los datos, los parámetros del escalador incorporan información estadística de las grabaciones que posteriormente se utilizarán para evaluar el modelo.

Aunque StandardScaler no utiliza directamente las etiquetas de clase para calcular esos parámetros, el conjunto de prueba deja de estar completamente aislado del preprocesamiento.

### ¿Qué consecuencias tiene?

La fuga de información puede ocasionar los siguientes problemas:

- **Evaluación poco confiable:** las métricas de prueba dejan de representar una evaluación completamente independiente.
- **Generalización deficiente:** el modelo puede comportarse peor al procesar audios nuevos.
- **Comparaciones injustas:** los resultados de distintos modelos pueden dejar de ser comparables si sus procedimientos de validación difieren.
- **Decisiones incorrectas:** un modelo puede seleccionarse por sus resultados de evaluación, aunque esos resultados no reflejen adecuadamente su desempeño fuera de los datos utilizados para desarrollarlo.

# 8. Modelado Paramétrico y No Paramétrico

---

## 8.1 Comparativa Teórica y Conceptual

| Criterio | Enfoque Paramétrico | Enfoque No Paramétrico |
| :--- | :--- | :--- |
| **Definición** | Asume que los datos provienen de una familia funcional de distribuciones teóricas con parámetros fijos $(\theta)$. | No impone supuestos a priori sobre la distribución subyacente; la complejidad crece con el volumen de muestras. |
| **Ventajas** | • Inferencia computacionalmente rápida ($O(K)$).<br>• Estructura compacta en almacenamiento.<br>• Menor riesgo de sobreajuste si la hipótesis distributiva es adecuada. | • Alta adaptabilidad a fronteras de decisión no lineales complejas.<br>• No fuerza supuestos geométricos artificiales sobre los datos acústicos. |
| **Desventajas y Limitaciones** | • Sesgo elevado (*high bias*) si la hipótesis distributiva es incorrecta.<br>• Rigidez estructural ante patrones atípicos. | • Alto consumo de memoria e inferencia costosa ($O(N)$).<br>• Vulnerable a la maldición de la dimensionalidad en espacios acústicos grandes. |

---

## 8.2 Estructura Operativa de los Enfoques

> ### Enfoque Paramétrico (Ej. GMM)
> * **Hipótesis:** Distribución teórica predefinida en el espacio acústico.
> * **Entrenamiento:** Aprende y almacena únicamente parámetros fijos:
>   * Pesos de mezcla ($w_k$)
>   * Medias ($\mu_k$)
>   * Matrices de covarianza ($\Sigma_k$)
> * **Inferencia:** Evaluación matemática compacta de complejidad $O(K)$. Los datos originales se descartan.

> ### Enfoque No Paramétrico (Ej. KNN)
> * **Hipótesis:** Libre de supuestos sobre la forma de la distribución.
> * **Entrenamiento:** No calcula parámetros globales; almacena la matriz de entrenamiento completa.
> * **Inferencia:** Evaluación dependiente de la base de datos de complejidad $O(N)$ calculando distancias métricas contra todas las muestras.

---

## 8.3 Casos de Estudio en Reconocimiento de Voz

### 1. Gaussian Mixture Models (GMM) — Enfoque Paramétrico
* **Fundamento:** Modela el espacio de coeficientes acústicos (ej. vectores MFCC) como una combinación lineal convexa de $K$ densidades gaussianas multivariadas:
  $$p(x) = \sum_{k=1}^K w_k \mathcal{N}(x \mid \mu_k, \Sigma_k) \quad \text{donde} \quad \sum_{k=1}^K w_k = 1, \; w_k \ge 0$$
* **Fase de entrenamiento:** Estima mediante el algoritmo de Esperanza-Maximización (EM) un conjunto finito de parámetros: pesos ($w_k$), medias ($\mu_k$) y matrices de covarianza ($\Sigma_k$).
* **Fase de inferencia:** Tras converger el entrenamiento, las muestras de audio originales se descartan; evaluar la probabilidad de una nueva trama solo requiere evaluar la función de densidad con los parámetros aprendidos.

### 2. $K$-Nearest Neighbors (KNN) — Enfoque No Paramétrico
* **Fundamento:** No estima funciones de densidad globales ni parámetros analíticos. El modelo consiste en retener en memoria la totalidad de los datos de entrenamiento.
* **Fase de inferencia:** Para clasificar una muestra de voz, computa distancias métricas (ej. euclidiana o Mahalanobis) contra todas las observaciones del conjunto y asigna la clase mayoritaria entre los $k$ vecinos más cercanos.
* **Limitación en voz:** Conforme el corpus escala a miles de horas o tramas acústicas, el costo de búsqueda se vuelve prohibitivo para sistemas de procesamiento continuo o en tiempo real.

# 9. Reproducibilidad y Validez Experimental

---

## 9.1 Flujo Metodológico de Particionamiento y Normalización

> **Paso 1: Separación por Grupos de Hablantes (Corpus Total)**
> * Se divide la base de datos a nivel sujeto:
>   * **Sujetos de entrenamiento:** Hablantes A, B, C, D, E
>   * **Sujetos de prueba:** Hablantes F, G (locutores nunca antes vistos)
>
> **Paso 2: Subdivisión de Entrenamiento y Ajuste**
> * El conjunto de entrenamiento se subdivide en:
>   * **Train Set:** Aprendizaje de pesos y representaciones acústicas.
>   * **Validation Set:** Sintonización y ajuste fino de hiperparámetros.
>
> **Paso 3: Estandarización y Prevención de Fuga de Datos (*Data Leakage*)**
> * `scaler.fit(X_train)` $\rightarrow$ Cálculo de media ($\mu$) y desviación estándar ($\sigma$) **exclusivamente** sobre el conjunto de entrenamiento.
> * `scaler.transform(X_test)` $\rightarrow$ Aplicación de la transformación con los parámetros aprendidos hacia la evaluación final, preservando la validez experimental.

---

## 9.2 Pilares Metodológicos en Señales de Voz

### 1. Particionamiento por Grupos de Hablantes (*Group / Speaker-Independent Split*)
* **Riesgo:** Si grabaciones de un mismo locutor se distribuyen indistintamente entre entrenamiento y prueba, el modelo aprende a identificar la identidad tímbrica del sujeto en lugar de los patrones acústicos o emocionales de interés.
* **Solución técnica:** La división debe realizarse agrupando estrictamente por sujeto (*Speaker-Independent*), garantizando que los hablantes de prueba no hayan sido escuchados durante la etapa de ajuste.

### 2. Control de Semillas Aleatorias (*Random Seeds*)
* **Aplicación:** Fijar explícitamente el estado del generador pseudoaleatorio (`numpy.random.seed`, `torch.manual_seed`, `random_state`) en:
  * La segmentación estratificada de datos.
  * La inicialización de pesos de modelos o algoritmos de agrupamiento.
  * Los procesos de submuestreo o aumento de datos (*data augmentation*).
* **Objetivo:** Asegurar el determinismo experimental para que cualquier evaluador obtenga resultados numéricos y matrices idénticas al ejecutar el código.

### 3. Fuga de Información (*Data Leakage*)
* **Definición:** Contaminación del proceso de aprendizaje debido a la inclusión inadvertida de información propia del conjunto de prueba.
* **Error crítico en procesamiento de audio:** Calcular media ($\mu$) y varianza ($\sigma^2$) de los descriptores sobre la totalidad del banco de datos antes de hacer la partición.
* **Protocolo estricto:** Ajustar los parámetros de escalamiento únicamente con el subconjunto de entrenamiento:
  $$\text{scaler.fit}(X_{\text{train}}) \implies \text{scaler.transform}(X_{\text{train}}), \; \text{scaler.transform}(X_{\text{test}})$$

### 4. Evaluación en Conjunto Cerrado (*Closed-Set*) vs. Abierto (*Open-Set*)
* **Closed-Set:** Asume que cualquier patrón de prueba pertenece obligatoriamente a una de las clases vistas previamente en el entrenamiento.
* **Open-Set:** El sistema cuenta con mecanismos de rechazo o análisis de incertidumbre para catalogar una muestra como "desconocida" o no perteneciente al corpus, simulando condiciones operativas reales.

### 5. Capacidad de Generalización
* Aptitud del sistema para sostener su rendimiento métrico (F1-score, exactitud, WER) ante condiciones no controladas de entorno:
  * Variabilidad fisiológica de nuevos hablantes (edad, género, acento).
  * Distorsión de canal por respuesta en frecuencia de diferentes micrófonos.
  * Presencia de ruido ambiental, reverberación y habla espontánea no estructurada.